# Regression models — our day-ahead residual-load forecast vs. SMARD

Implements [`.claude/specs/05-regression-models.md`](../../.claude/specs/05-regression-models.md).

We forecast `residual_load` for every hour of a delivery day `DAY`, **issued at 18:00 on `DAY−1`**: the
same point in time as SMARD's public day-ahead forecast. Every model is scored against SMARD's
`fc_residual_load` and against a seasonal-naive floor, on **identical hours**.

**Our models are post-processors of SMARD's forecast.** They take SMARD's published component
forecasts (`fc_grid_load`, `fc_gen_wind_solar`) as inputs. If one wins, the honest claim is "we
reduce SMARD's error by X %", not "we forecast better than the TSOs".

## How to use this notebook

- Change a value in the configuration cells (§1.3), then re-run top to bottom. No later cell
  hardcodes a value the configuration holds.
- The interpretation is written **once**, in the closing section, for the default configuration.
  Tables and plots in between carry titles and units, no commentary.

## What this notebook produces

- day-ahead forecasts from seasonal naive `DAY−7`, `sarimax_fourier` and LightGBM direct / hybrid
  (XGBoost direct / hybrid when switched on), each under a **static** and a **rolling** split method
  on the same test year
- an empirical 95 % prediction interval per model, with its measured coverage
- one scoreboard (accuracy, extremes, intervals) including SMARD and seasonal naive
- two optional exports in `data/models/`, written only when `EXPORT_ENABLED` is on (default off)

## Conventions

- **Sign:** `error = forecast − actual`, as in spec 04. **Positive = over-forecast.**
- **Units:** hourly readings, forecasts and errors in `MWh`; capacity in `MW`; skill and coverage in
  `%`. The `MW` relabelling in `team-EDA.ipynb` does not apply here.
- **Durations, never row counts.** No literal calendar year or date appears in code.
- `time_series` holds exactly `SERIES + DERIVED`. Everything this notebook builds lives in separate
  frames.

## Not in this notebook

- Holt-Winters, seasonal ARIMA with a period `m`, MAPE, weather data, `fc_residual_load` as a feature
- risk flags on our forecast (parked [04.3](../../.claude/specs/04.3-risk-label-link.md)) and the
  remaining baselines of parked [04.1](../../.claude/specs/04.1-naive-baseline.md)
- significance tests, MLflow, any change to `modeling/`, reBAP

---

## 1 Setup and configuration

### 1.1 Shared setup (inherited)

Same setup as [`team-EDA.ipynb`](../01_eda/team-EDA.ipynb) §1, as reused by
[`risk-definition.ipynb`](../03_risk_classification/risk-definition.ipynb) and
[`forecast-metrics-claude.ipynb`](../02_forecast_metrics/forecast-metrics-claude.ipynb). Inherited,
not re-derived:

- the data-directory resolver (walks **upward** from the working directory)
- loading, renaming, the German-CSV float conversion and the dtype asserts
- `time_series`, `SERIES`, `DERIVED`, `YEARS`, `DAY_NAMES`, the season mapping
- `style_timeseries` (`ylabel` required)
- the duration pattern and the day-completeness rule from `risk-definition.ipynb` §3.1

Not needed here, so not repeated: `_complete_periods`, `period_mean`, `period_energy`,
`seasonal_plot` and the team-EDA colour configuration. Model colours live in the registry (§1.3).

`data/` is **gitignored**, so `data/smard.csv` does not come with a clone. Regenerate it with
`notebooks/API-connection.ipynb`.

In [ ]:
import itertools
import time
import warnings
from pathlib import Path

import holidays
import lightgbm as lgb
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
import xgboost as xgb
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import kpss

# Walk up from the working directory to the first parent holding a `data/` folder
DATA_DIR = next(
    (p / "data" for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(
        f"no data/ directory found in {Path.cwd()} or any parent — start the kernel inside the "
        "repository, then re-run."
    )
DATA = DATA_DIR / "smard.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running notebooks/API-connection.ipynb top to bottom."
    )

print(
    f"pandas {pd.__version__} · numpy {np.__version__} · statsmodels {statsmodels.__version__} · "
    f"lightgbm {lgb.__version__} · xgboost {xgb.__version__}"
)
print(f"Data directory: {DATA}")

In [ ]:
def style_timeseries(ax, title, ylabel):
    """Custom grid, no box, year ticks.

    `ylabel` is required: every plot must state its unit.
    """
    ax.set_title(
        title,
        loc="center",
        fontsize=15,
        pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        ylabel,
        color="grey"
    )
    ax.grid(
        axis="y",
        color="0.9",
        linewidth=0.8
    )
    ax.set_axisbelow(True)

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    ax.tick_params(
        colors="black",
        length=0  # hide ticks of values
    )
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")

In [ ]:
# The CSV headers exactly as notebooks/API-connection.ipynb writes them.
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid Load": "fc_grid_load",
    "Forecast Residual Load": "fc_residual_load",
    "Capacity Wind Offshore": "cap_wind_off",
    "Capacity Wind Onshore": "cap_wind_on",
    "Capacity Solar": "cap_solar"
}

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")

assert set(raw.columns) == {"timestamp"} | set(COLUMNS), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | set(COLUMNS)))}"
)

raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

time_series = raw.set_index("timestamp").sort_index()
del raw  # the flat frame does not outlive the loading cell

# Positive is_float_dtype test, not `!= object`: under pandas 3 an unconverted German-decimal
# column lands as StringDtype, and `!= object` would wave it straight through.
assert all(
    pd.api.types.is_float_dtype(time_series[c]) for c in COLUMNS.values()
), time_series.dtypes

# Snapshot taken before any other cell can touch the frame, so the closing self-check can prove
# nothing in between mutated it.
LOADED = {
    "rows": len(time_series),
    "start": time_series.index.min(),
    "end": time_series.index.max(),
}

print(f"shape           : {time_series.shape[0]:,} rows x {time_series.shape[1]} columns")
print(f"index           : {time_series.index.min()}  ->  {time_series.index.max()}")
print(
    f"index monotonic : {time_series.index.is_monotonic_increasing}, "
    f"unique: {time_series.index.is_unique}"
)

In [ ]:
SERIES = [
    "wind_off", "wind_on", "solar", "grid_load", "residual_load",
    "fc_gen_wind_solar", "fc_grid_load", "fc_residual_load",
    "cap_wind_off", "cap_wind_on", "cap_solar"
]

DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

time_series["renewables"] = time_series[["wind_on", "wind_off", "solar"]].sum(axis=1)
time_series["year"] = time_series.index.year
time_series["month"] = time_series.index.month
time_series["hour"] = time_series.index.hour
time_series["dow"] = time_series.index.dayofweek
time_series["is_weekend"] = time_series.index.dayofweek >= 5
time_series["date"] = time_series.index.date
time_series["season"] = pd.Categorical(
    time_series.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
time_series["season_year"] = time_series.index.year + (time_series.index.month == 12)

# Outputs True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
time_series["spans_gap"] = time_series.index.to_series().diff() > pd.Timedelta("1h")

DERIVED = [
    "renewables", "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

# Plain ints, not np.int32: they end up in titles, labels and dict keys all over the notebook.
YEARS = sorted(int(y) for y in time_series["year"].unique())

print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {time_series.shape[1]} columns")
print(f"YEARS = {YEARS}")

### 1.2 Resolution and durations

Every window, lag, refit interval, Fourier period, issue time and publication lag in this notebook is
a **duration**, converted to an observation count from the **measured** resolution by `n_obs`. A
switch to SMARD's quarter-hour data would change the counts, not the definitions.

At a sub-hourly resolution, models would be compared with each other at native resolution, and with
SMARD only after aggregating to hourly means, because `smard_forecast_errors_hourly.csv` holds
hourly errors. This notebook runs on the hourly `data/smard.csv` only.

In [ ]:
# Resolution is measured, not assumed.
RESOLUTION = time_series.index.to_series().diff().mode().iloc[0]

DAY_COMPLETENESS = 23 / 24   # accepts the spring-DST day, rejects materially short days


def n_obs(duration):
    """Observation count of `duration` at the measured resolution. Refuses a non-multiple."""
    count = duration / RESOLUTION
    if count != int(count):
        raise ValueError(f"{duration} is not a whole multiple of the resolution {RESOLUTION}")
    return int(count)


def describe(value):
    """Short readable form of a configuration value: hours below two days, whole days above."""
    if isinstance(value, pd.Timedelta):
        if value >= pd.Timedelta(days=2) and value == pd.Timedelta(days=value.days):
            return f"{value.days} days"
        return f"{value / pd.Timedelta(hours=1):g} h"
    if isinstance(value, pd.DateOffset):
        return ", ".join(f"{n} {unit.rstrip('s') if n == 1 else unit}" for unit, n in value.kwds.items())
    return str(value)


EXPECTED_OBS_PER_DAY = n_obs(pd.Timedelta(days=1))
MIN_OBS_PER_DAY = int(np.ceil(DAY_COMPLETENESS * EXPECTED_OBS_PER_DAY))

print(f"resolution        : {describe(RESOLUTION)}  ->  {EXPECTED_OBS_PER_DAY} observations per full day")
print(f"day completeness  : >= {MIN_OBS_PER_DAY} of {EXPECTED_OBS_PER_DAY} observations")

### 1.3 Configuration

Every value a later cell depends on is set here and nowhere else. Edit these the way you edit a
colour map, then re-run the notebook top to bottom. The defaults are the spec's.

| Cell | Holds |
|---|---|
| `DATA_INFO` | the forecast setting: issue time on `DAY−1`, the actuals publication lag, the capacity publication rule |
| `WINDOWS`, `TRAIN_TEST_SPLIT_METHOD` | test, validation and training window lengths, the refit interval, which split methods run |
| `MODELS` | the model registry: per model an on/off switch, family, architecture, fixed parameters, tuning grid, label and colour |
| `FEATURES` | one switch per booster feature group (**provisional**) and the durations the groups use |
| `INTERVAL`, `PLOT_MODEL`, `PLOT_SPLIT_METHOD`, `EXPORT_ENABLED` | interval level, forecast-plot selection, export toggle |

Windows and times are durations: `pd.Timedelta`, or `pd.DateOffset` where calendar months are meant.

In [ ]:
DATA_INFO = {
    "issue_days_before": pd.Timedelta(days=1),     # the forecast for DAY is issued on DAY−1 ...
    "issue_clock": pd.Timedelta(hours=18),         # ... at 18:00 local, once both SMARD components are out
    "actuals_lag": pd.Timedelta(hours=3),          # smard.de shows actuals ~2 h behind real time, +1 h margin
    "capacity_published_after": pd.Timedelta(0),   # year Y's cap_* value: 1 January of Y, 00:00
}

WINDOWS = {
    "test": pd.Timedelta(days=365),         # the last 365 complete delivery days
    "validation": pd.Timedelta(days=365),   # the 365 delivery days before the test window
    "train": pd.DateOffset(months=24),      # calendar months, not 730 days
    "refit_every": pd.Timedelta(days=30),   # rolling method only
}

TRAIN_TEST_SPLIT_METHOD = {"static": True, "rolling": True}

# Clock times implied by DATA_INFO, printed so the rule can be read without doing the arithmetic.
cutoff_clock = DATA_INFO["issue_clock"] - DATA_INFO["actuals_lag"]
last_actual_clock = cutoff_clock - RESOLUTION
first_capacity_use = pd.Timestamp(year=YEARS[0], month=1, day=1) + DATA_INFO["capacity_published_after"]

print("DATA_INFO")
print(
    f"  issue time            : {pd.Timestamp(0) + DATA_INFO['issue_clock']:%H:%M} on "
    f"DAY−{DATA_INFO['issue_days_before'].days}"
)
print(
    f"  actuals lag           : {describe(DATA_INFO['actuals_lag'])} -> availability cutoff "
    f"{pd.Timestamp(0) + cutoff_clock:%H:%M}, last usable actual ROW stamped "
    f"{pd.Timestamp(0) + last_actual_clock:%H:%M} on DAY−{DATA_INFO['issue_days_before'].days}"
)
print(
    f"  capacity publication  : {describe(DATA_INFO['capacity_published_after'])} after the start of "
    f"its year (the {YEARS[0]} value is usable from {first_capacity_use:%Y-%m-%d %H:%M})"
)
print("WINDOWS")
for name, value in WINDOWS.items():
    print(f"  {name:<22}: {describe(value)}")
print(f"TRAIN_TEST_SPLIT_METHOD : {TRAIN_TEST_SPLIT_METHOD}")

In [ ]:
SEED = 42  # every booster and every random draw in this notebook

# One grid and one parameter set per booster, shared by its direct and hybrid entries: a hybrid is
# tuned over exactly the same configurations as its direct variant.
LGBM_PARAMS = {"learning_rate": 0.05, "random_state": SEED, "verbose": -1}
LGBM_GRID = {"num_leaves": [31, 63], "n_estimators": [300, 800]}
XGB_PARAMS = {"learning_rate": 0.05, "random_state": SEED, "tree_method": "hist"}
XGB_GRID = {"max_depth": [4, 6], "n_estimators": [300, 800]}

MODELS = {
    "sarimax_fourier": {
        "enabled": False,
        "family": "sarimax",
        "architecture": None,
        "params": {
            "order": (1, 0, 1),                  # fixed, no order search; d = 0 (see the KPSS check)
            "trend": "c",                        # intercept of the regression
            "fourier": {pd.Timedelta(days=1): 4, pd.Timedelta(days=7): 3},  # period -> order K
            "exog": ["fc_grid_load", "fc_gen_wind_solar", "holiday"],
        },
        "grid": {},
        "label": "SARIMAX + Fourier",
        "color": "#D9A53A",
    },
    "lgbm_direct": {
        "enabled": True,
        "family": "lightgbm",
        "architecture": "direct",
        "params": LGBM_PARAMS,
        "grid": LGBM_GRID,
        "label": "LightGBM direct",
        "color": "#2C6EBA",
    },
    "lgbm_hybrid": {
        "enabled": True,
        "family": "lightgbm",
        "architecture": "hybrid",
        "params": LGBM_PARAMS,
        "grid": LGBM_GRID,
        "label": "LightGBM hybrid",
        "color": "#2F8F5B",
    },
    "xgb_direct": {
        "enabled": False,
        "family": "xgboost",
        "architecture": "direct",
        "params": XGB_PARAMS,
        "grid": XGB_GRID,
        "label": "XGBoost direct",
        "color": "#E95D0F",
    },
    "xgb_hybrid": {
        "enabled": False,
        "family": "xgboost",
        "architecture": "hybrid",
        "params": XGB_PARAMS,
        "grid": XGB_GRID,
        "label": "XGBoost hybrid",
        "color": "#B10F0F",
    },
}

# Rows outside the registry, with fixed colours. SMARD is drawn dashed.
FIXED = {
    "actual": {"label": "Actual residual load", "color": "#1C1C1C"},
    "smard": {"label": "SMARD day-ahead", "color": "#48505A"},
    "seasonal_naive": {"label": "Seasonal naive (DAY−7)", "color": "#9098A2", "lag": pd.Timedelta(days=7)},
}

registry = pd.DataFrame(
    {
        key: {
            "enabled": m["enabled"],
            "family": m["family"],
            "architecture": m["architecture"] or "—",
            "grid configurations": int(np.prod([len(v) for v in m["grid"].values()])),
            "grid": m["grid"] or "—",
            "label": m["label"],
            "color": m["color"],
        }
        for key, m in MODELS.items()
    }
).T
print(f"MODELS: {sum(m['enabled'] for m in MODELS.values())} of {len(MODELS)} enabled, seed {SEED}")
display(registry)
print("fixed parameters")
for key, m in MODELS.items():
    params = {k: ({describe(p): o for p, o in v.items()} if k == "fourier" else v) for k, v in m["params"].items()}
    print(f"  {key:<16}: {params}")
print(f"outside the registry: {', '.join(FIXED)}")

In [ ]:
# Booster feature groups (spec Behaviour 18). PROVISIONAL: the team's feature-engineering work may
# replace them. They feed the direct boosters and the hybrids' stage 2 only. SARIMAX's inputs sit in
# its registry entry, and the hybrids' stage 1 always uses the two SMARD forecasts plus a trend.
FEATURES = {
    "calendar": True,             # local hour, day of week, month, is_weekend, holiday flag
    "smard_forecast": True,       # fc_grid_load, fc_gen_wind_solar for the target hour
    "lags": True,                 # residual_load at the same local hour on DAY−2 and DAY−7; last actual at the cutoff
    "recent_smard_error": True,   # mean err_grid_load and err_renewables over the window ending at the cutoff
    "capacity": True,             # cap_wind_off + cap_wind_on + cap_solar under the publication rule
}

FEATURE_WINDOWS = {
    "same_hour_lags": [pd.Timedelta(days=2), pd.Timedelta(days=7)],
    "recent_smard_error": pd.Timedelta(hours=24),
}

print("FEATURES (provisional)")
for group, enabled in FEATURES.items():
    print(f"  {group:<20}: {'on' if enabled else 'off'}")
print("FEATURE_WINDOWS")
for name, value in FEATURE_WINDOWS.items():
    shown = ", ".join(describe(v) for v in value) if isinstance(value, list) else describe(value)
    print(f"  {name:<20}: {shown}")

In [ ]:
INTERVAL = {"level": 0.95}      # empirical prediction interval, calibrated on the validation year

PLOT_MODEL = None               # None: the registry model with the lowest test MAE; or a key, e.g. "lgbm_hybrid"
PLOT_SPLIT_METHOD = "rolling"   # falls back to "static" when rolling is switched off

EXPORT_ENABLED = False          # True writes data/models/model_*.csv; the folder is not created here

print(f"INTERVAL          : {INTERVAL['level']:.0%} prediction interval")
print(f"PLOT_MODEL        : {PLOT_MODEL if PLOT_MODEL else 'None -> lowest test MAE among registry models'}")
print(f"PLOT_SPLIT_METHOD : {PLOT_SPLIT_METHOD}")
print(f"EXPORT_ENABLED    : {EXPORT_ENABLED}")

### 1.4 SMARD's hourly errors

`data/metrics/smard_forecast_errors_hourly.csv` is written by
[`forecast-metrics-claude.ipynb`](../02_forecast_metrics/forecast-metrics-claude.ipynb) (spec 04). It
is used twice:

- to **re-score SMARD** on this notebook's common hours, instead of recomputing SMARD from
  `data/smard.csv`
- as the source of the `recent_smard_error` feature group (`err_grid_load`, `err_renewables`)

It stays in its own frame, `smard_errors`, and is never merged into `time_series`. The cell stops if
the file is missing, or if its timestamps differ from `time_series.index`: that would be an export
from another snapshot of `data/smard.csv`.

In [ ]:
SMARD_ERRORS = DATA_DIR / "metrics" / "smard_forecast_errors_hourly.csv"
SMARD_ERRORS_SOURCE = "notebooks/02_forecast_metrics/forecast-metrics-claude.ipynb"
SMARD_ERROR_COLUMNS = ["residual_load", "fc_residual_load", "err_residual_load", "err_grid_load", "err_renewables"]

if not SMARD_ERRORS.exists():
    raise FileNotFoundError(
        f"{SMARD_ERRORS} not found. data/metrics/ is gitignored, so the file is not in a fresh clone — "
        f"regenerate it by running {SMARD_ERRORS_SOURCE} top to bottom."
    )

smard_errors = pd.read_csv(SMARD_ERRORS)
smard_errors["timestamp"] = pd.to_datetime(smard_errors["timestamp"], format="%Y-%m-%d %H:%M:%S")
smard_errors = smard_errors.set_index("timestamp")

missing_columns = sorted(set(SMARD_ERROR_COLUMNS) - set(smard_errors.columns))
if missing_columns:
    raise ValueError(f"{SMARD_ERRORS.name} lacks {missing_columns} — re-run {SMARD_ERRORS_SOURCE}.")

if not smard_errors.index.equals(time_series.index):
    raise ValueError(
        f"{SMARD_ERRORS.name} does not match data/smard.csv: "
        f"{len(smard_errors.index.difference(time_series.index)):,} timestamps only in the errors file, "
        f"{len(time_series.index.difference(smard_errors.index)):,} only in smard.csv "
        f"(errors file {smard_errors.index.min()} -> {smard_errors.index.max()}). It is a stale export "
        f"from another snapshot — re-run {SMARD_ERRORS_SOURCE} top to bottom."
    )

print(f"smard_errors    : {len(smard_errors):,} rows, index identical to time_series")
print(f"range           : {smard_errors.index.min()}  ->  {smard_errors.index.max()}")
print(f"columns used    : {SMARD_ERROR_COLUMNS}")

### 1.5 Initial self-check

Structural checks on the loaded data and on the configuration, free of any hardcoded row count or
date. The closing self-check re-runs the data invariants and compares against `LOADED`.

In [ ]:
assert all(pd.api.types.is_float_dtype(time_series[c]) for c in SERIES), time_series[SERIES].dtypes
assert time_series.index.is_monotonic_increasing, "index is not sorted"
assert time_series.index.is_unique, "index has duplicate timestamps"
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert YEARS == sorted(int(y) for y in time_series["year"].unique())

# The configuration is internally consistent.
REGISTRY_KEYS = {"enabled", "family", "architecture", "params", "grid", "label", "color"}
for key, m in MODELS.items():
    assert REGISTRY_KEYS <= set(m), f"{key}: missing {REGISTRY_KEYS - set(m)}"
    assert m["family"] in {"sarimax", "lightgbm", "xgboost"}, (key, m["family"])
    if m["family"] == "sarimax":
        assert "fc_residual_load" not in m["params"]["exog"], f"{key}: fc_residual_load is never an input"
    else:
        assert m["architecture"] in {"direct", "hybrid"}, (key, m["architecture"])
    if m["architecture"] == "hybrid":
        direct = [d for d in MODELS.values() if d["family"] == m["family"] and d["architecture"] == "direct"]
        assert direct and m["grid"] == direct[0]["grid"], f"{key}: a hybrid uses its direct variant's grid"
assert not set(MODELS) & set(FIXED), "a registry key collides with a row outside the registry"
assert any(m["enabled"] for m in MODELS.values()), "switch on at least one registry model"
assert set(TRAIN_TEST_SPLIT_METHOD) == {"static", "rolling"}, TRAIN_TEST_SPLIT_METHOD
assert any(TRAIN_TEST_SPLIT_METHOD.values()), "switch on at least one split method"
assert PLOT_MODEL is None or MODELS.get(PLOT_MODEL, {}).get("enabled"), f"PLOT_MODEL {PLOT_MODEL!r} is not an enabled registry key"
assert PLOT_SPLIT_METHOD in TRAIN_TEST_SPLIT_METHOD, PLOT_SPLIT_METHOD
assert set(FEATURES) == {"calendar", "smard_forecast", "lags", "recent_smard_error", "capacity"}, FEATURES
assert 0 < INTERVAL["level"] < 1, INTERVAL
assert isinstance(EXPORT_ENABLED, bool)

print("setup self-check passed")
print(f"  {len(SERIES)} float series, index sorted and unique, columns == SERIES + DERIVED")
print(f"  {LOADED['rows']:,} rows, {LOADED['start']} -> {LOADED['end']}, years {YEARS}")
print(f"  configuration consistent: {sum(m['enabled'] for m in MODELS.values())} registry models, "
      f"split methods {[k for k, v in TRAIN_TEST_SPLIT_METHOD.items() if v]}")

---

## 2 Forecast setting

Every model and every training row obeys one rule for **what is known when**. Defaults from
`DATA_INFO` in brackets:

| Item | Rule |
|---|---|
| `ISSUE_TIME` | the forecast for delivery day `DAY` is issued on `DAY−1` (18:00 local) |
| `AVAILABILITY_CUTOFF` | `ISSUE_TIME − actuals lag` (3 h, so 15:00 on `DAY−1`) |
| actuals | a row stamped `t` (interval start) is usable if `t + resolution ≤ AVAILABILITY_CUTOFF`: up to the row stamped 14:00 |
| SMARD forecasts for `DAY` | available: both components are published by 18:00 on `DAY−1` |
| SMARD forecasts for the rest of `DAY−1` | available: published on `DAY−2` |
| target hours | every local hour of `DAY`: 24, or 23 on the spring DST day (the autumn fold is collapsed by SMARD) |
| training rows | delivery days whose target hours are all observed by the fit's cutoff, each row built as it would have been at its own issue time |

`ISSUE_TIME` and `AVAILABILITY_CUTOFF` are **per-day timestamps** from `forecast_setting`, never
constants. The truncation leakage test that proves the rule holds needs the feature builder and
the fitted models, so it runs in §5.4.

### 2.1 The forecast-setting helper

`forecast_setting(days)` returns, per delivery day, the issue time, the availability cutoff and the
target hours. `SETTING` holds it for every day of the record, `ROW_SETTING` the same times per row.

In [ ]:
# Delivery day of every row: its local calendar date as a local midnight (the DERIVED "date" boundary).
DAY_OF_ROW = time_series.index.normalize()
TARGET_HOURS = time_series.index.groupby(DAY_OF_ROW)   # delivery day -> its local hours in the record
DAYS = pd.DatetimeIndex(sorted(TARGET_HOURS))


def forecast_setting(days):
    """The forecast setting of each delivery day in `days` (local midnights).

    One row per day: `issue_time` (ISSUE_TIME), `cutoff` (AVAILABILITY_CUTOFF) and `target_hours`,
    the day's local hours as they exist in the record. Every model, feature and training row takes
    its times from here.
    """
    days = pd.DatetimeIndex(days)
    issue_time = days - DATA_INFO["issue_days_before"] + DATA_INFO["issue_clock"]
    return pd.DataFrame(
        {
            "issue_time": issue_time,
            "cutoff": issue_time - DATA_INFO["actuals_lag"],
            "target_hours": [TARGET_HOURS[day] for day in days],
        },
        index=days,
    )


def available(stamps, cutoff):
    """True where an observation stamped `stamps` (interval start) has ended by `cutoff`."""
    return stamps + RESOLUTION <= cutoff


SETTING = forecast_setting(DAYS)
ROW_SETTING = pd.DataFrame(
    {
        "day": DAY_OF_ROW,
        "issue_time": SETTING["issue_time"].reindex(DAY_OF_ROW).to_numpy(),
        "cutoff": SETTING["cutoff"].reindex(DAY_OF_ROW).to_numpy(),
    },
    index=time_series.index,
)

hours_per_day = SETTING["target_hours"].map(len)
dst_days = SETTING.index[hours_per_day < EXPECTED_OBS_PER_DAY]


def setting_summary(day):
    """One readable row of the forecast setting for delivery day `day`."""
    row = SETTING.loc[day]
    last_actual = row["cutoff"] - RESOLUTION
    # Counted in rows, not clock hours: the missing spring-DST hour shortens the horizon by one row.
    last_position = time_series.index.searchsorted(last_actual, side="right") - 1
    rows_ahead = time_series.index.get_indexer(row["target_hours"]) - last_position
    return {
        "DAY": f"{day:%Y-%m-%d} ({DAY_NAMES[day.dayofweek]})",
        "ISSUE_TIME": f"{row['issue_time']:%Y-%m-%d %H:%M}",
        "AVAILABILITY_CUTOFF": f"{row['cutoff']:%Y-%m-%d %H:%M}",
        "last usable actual row": f"{last_actual:%Y-%m-%d %H:%M}",
        "target hours": len(row["target_hours"]),
        "rows after last usable actual": f"{rows_ahead.min()}–{rows_ahead.max()}",
    }


print(f"delivery days     : {len(SETTING):,}  ({DAYS[0]:%Y-%m-%d} .. {DAYS[-1]:%Y-%m-%d})")
print(f"target hours/day  : {hours_per_day.value_counts().sort_index(ascending=False).to_dict()}")
print(f"spring DST days   : {len(dst_days)}, each with {hours_per_day[dst_days].min()} target hours")
print("\nForecast setting of two delivery days: the record's latest spring DST day and its last day")
display(pd.DataFrame([setting_summary(dst_days[-1]), setting_summary(DAYS[-1])]).set_index("DAY"))

### 2.2 Feature availability rules

| Input | Usable at `ISSUE_TIME` | Why |
|---|---|---|
| actuals: `residual_load`, and SMARD's errors (they need the actual) | rows with `t + resolution ≤ AVAILABILITY_CUTOFF` | actuals appear on smard.de about 2 h late |
| SMARD component forecasts for the target hour | yes | published by 18:00 on `DAY−1` |
| SMARD component forecasts for `DAY−1` from the cutoff on | yes | published on `DAY−2`; SARIMAX's exogenous inputs |
| calendar: hour, weekday, month, weekend, holiday | always | known in advance |
| installed capacity | the latest value published by `ISSUE_TIME` | publication rule, §2.3 |

The cell below checks the same-hour lags in `FEATURE_WINDOWS` against the rule, for every row of the
record. A lag that reaches past the cutoff (e.g. `DAY−1`, whose afternoon and evening come after
15:00) stops the notebook instead of leaking silently. The two windows that end at the cutoff hold by
definition; the leakage test (§5.4) checks every built feature.

In [ ]:
cutoffs = ROW_SETTING["cutoff"].to_numpy()
issue_day = f"DAY−{DATA_INFO['issue_days_before'].days}"
availability = []

for lag in FEATURE_WINDOWS["same_hour_lags"]:
    source = time_series.index - lag
    usable = available(source, cutoffs)
    if not usable.all():
        raise ValueError(
            f"same-hour lag {describe(lag)} reaches past the availability cutoff for "
            f"{(~usable).sum():,} rows: it would leak. Use a lag the forecast setting allows."
        )
    availability.append({
        "feature": f"residual_load at the same local hour, {describe(lag)} earlier",
        "source rows": "one per target hour",
        "smallest margin to the cutoff": describe((cutoffs - (source + RESOLUTION)).min()),
    })

# These two windows are defined to end at the cutoff; the leakage test confirms the built features do.
for feature, window in [
    ("last residual_load before the cutoff", RESOLUTION),
    ("mean err_grid_load and err_renewables", FEATURE_WINDOWS["recent_smard_error"]),
]:
    availability.append({
        "feature": feature,
        "source rows": (
            f"{describe(window)} ending at the cutoff, last row "
            f"{pd.Timestamp(0) + cutoff_clock - RESOLUTION:%H:%M} on {issue_day}"
        ),
        "smallest margin to the cutoff": describe(pd.Timedelta(0)),
    })

print("Actual-derived features against the availability rule (lags checked on every row of the record)")
display(pd.DataFrame(availability).set_index("feature"))

### 2.3 Capacity publication rule

The `cap_*` columns hold one value per calendar year. The project treats year `Y`'s value as
**published on 1 January of `Y`, 00:00** (team decision, `DATA_INFO`). A delivery day uses the
latest value published at or before its issue time:

- Most delivery days use the **current** calendar year's value.
- **1 January uses the previous year's value.** It is issued at 18:00 on 31 December, before the new
  year's value is out. This is why the rule is a publication time, not a fixed year offset.
- Only the record's first day is issued before the first publication and has **no** capacity value.
  It is unforecastable anyway (§2.4), and the default 24-month windows never reach it.

Growth of the fleet within a year is invisible in a yearly step value.

In [ ]:
CAP_COLUMNS = ["cap_wind_off", "cap_wind_on", "cap_solar"]


def capacity_table(frame):
    """One row per calendar year in `frame`: its cap_* values, their total (MW), and when they count as published."""
    per_year = frame[CAP_COLUMNS].groupby(frame.index.year)
    if (per_year.nunique() > 1).any().any():
        raise ValueError("a cap_* column changes within a calendar year: the yearly-step assumption fails")
    table = per_year.first().rename_axis("capacity_year")
    table["cap_total"] = table[CAP_COLUMNS].sum(axis=1)
    table["published_at"] = [
        pd.Timestamp(year=year, month=1, day=1) + DATA_INFO["capacity_published_after"] for year in table.index
    ]
    return table


def published_capacity(issue_times, table):
    """Capacity year and total (MW) of the latest value published at or before each issue time.

    `issue_times` must be sorted. Empty (NaN) before the first publication. Rows keep the input order.
    """
    left = pd.DataFrame({"issue_time": pd.DatetimeIndex(issue_times)})
    right = table.reset_index()[["published_at", "capacity_year", "cap_total"]]
    # merge_asof refuses mixed datetime units, and Timestamp arithmetic can change the unit.
    right["published_at"] = right["published_at"].astype(left["issue_time"].dtype)
    merged = pd.merge_asof(
        left,
        right,
        left_on="issue_time",
        right_on="published_at",
        direction="backward",
    )
    return merged[["capacity_year", "cap_total"]].astype({"capacity_year": "Int64"})


CAPACITY = capacity_table(time_series)
DAY_CAPACITY = published_capacity(SETTING["issue_time"], CAPACITY).set_axis(SETTING.index)

shown = CAPACITY.assign(
    published_at=CAPACITY["published_at"].dt.strftime("%Y-%m-%d %H:%M"),
    used_in_record=CAPACITY["published_at"] <= SETTING["issue_time"].max(),
)
print("Installed capacity per year (MW) and its publication time")
display(shown.style.format("{:,.0f}", subset=CAP_COLUMNS + ["cap_total"]))

# The record's latest 1 January and its two neighbours, derived from the data.
new_year = DAYS[(DAYS.month == 1) & (DAYS.day == 1)][-1]
around = [new_year - pd.Timedelta(days=1), new_year, new_year + pd.Timedelta(days=1)]
example = pd.DataFrame(
    {
        "ISSUE_TIME": SETTING.loc[around, "issue_time"].dt.strftime("%Y-%m-%d %H:%M"),
        "capacity year used": DAY_CAPACITY.loc[around, "capacity_year"],
        "years before DAY's year": pd.Index(around).year - DAY_CAPACITY.loc[around, "capacity_year"],
        "cap_total (MW)": DAY_CAPACITY.loc[around, "cap_total"].map("{:,.0f}".format),
    },
    index=pd.Index([f"{d:%Y-%m-%d}" for d in around], name="DAY"),
)
print(f"\nThe 1 January consequence, around the record's latest new year ({new_year:%Y-%m-%d})")
display(example)

no_capacity = SETTING.index[DAY_CAPACITY["capacity_year"].isna()]
print(f"\ndelivery days without a published capacity value: {len(no_capacity):,}", end="")
print(f" ({no_capacity[0]:%Y-%m-%d} .. {no_capacity[-1]:%Y-%m-%d})" if len(no_capacity) else "")

### 2.4 Unforecastable days

`DAY` cannot be forecast, by **any** model, as soon as a SMARD component forecast (`fc_grid_load`,
`fc_gen_wind_solar`) is missing for an hour of `DAY`, or for an hour of `DAY−1` from the cutoff on:
SARIMAX takes those hours as exogenous inputs. Such a day has no input rows. It is a **data
exclusion**, not a model failure: it is left out of training and scoring and counted here, from the
data. The record's first day is excluded for the same reason: its `DAY−1` lies outside the record.

In [ ]:
FORECAST_INPUTS = ["fc_grid_load", "fc_gen_wind_solar"]

# Cumulative count of rows lacking a component forecast: any [start, end) row range is then one subtraction.
missing_input = time_series[FORECAST_INPUTS].isna().any(axis=1).to_numpy()
cum_missing = np.concatenate([[0], np.cumsum(missing_input)])

first_unavailable = time_series.index.searchsorted(SETTING["cutoff"])   # first DAY−1 row after the cutoff
day_start = time_series.index.searchsorted(SETTING.index)
day_end = time_series.index.searchsorted(SETTING.index + pd.Timedelta(days=1))

EXCLUSION = pd.DataFrame(
    {
        "forecast missing on DAY": cum_missing[day_end] - cum_missing[day_start] > 0,
        "forecast missing on DAY−1 after the cutoff": cum_missing[day_start] - cum_missing[first_unavailable] > 0,
        "DAY−1 outside the record": SETTING["cutoff"] < time_series.index[0],
    },
    index=SETTING.index,
)
FORECASTABLE = ~EXCLUSION.any(axis=1)

excluded = EXCLUSION[~FORECASTABLE]
print(f"forecastable delivery days : {FORECASTABLE.sum():,} of {len(FORECASTABLE):,}")
print(f"unforecastable             : {len(excluded)}")
display(
    excluded.apply(lambda row: ", ".join(row.index[row]), axis=1)
    .rename("reason")
    .set_axis(pd.Index([f"{d:%Y-%m-%d}" for d in excluded.index], name="DAY"))
    .to_frame()
)

### 2.5 Self-check

In [ ]:
assert SETTING.index.equals(DAYS) and DAYS.is_unique and DAYS.is_monotonic_increasing
assert (SETTING["issue_time"] - SETTING["cutoff"] == DATA_INFO["actuals_lag"]).all()
assert (SETTING["issue_time"] < SETTING.index).all(), "a forecast is issued after its delivery day starts"
assert sum(map(len, SETTING["target_hours"])) == len(time_series), "target hours do not partition the record"
assert all((hours.normalize() == day).all() for day, hours in SETTING["target_hours"].items())
assert ROW_SETTING.index.equals(time_series.index) and ROW_SETTING.notna().all().all()

# Capacity: never a value published after the issue time, never from a later calendar year.
used = DAY_CAPACITY["capacity_year"].notna().to_numpy()
published_at = CAPACITY.loc[DAY_CAPACITY["capacity_year"][used], "published_at"].to_numpy()
assert (published_at <= SETTING["issue_time"].to_numpy()[used]).all(), "capacity used before its publication"
assert (DAY_CAPACITY["capacity_year"][used].to_numpy() <= SETTING.index.year[used]).all()

assert EXCLUSION.index.equals(SETTING.index) and FORECASTABLE.dtype == bool
assert list(time_series.columns) == SERIES + DERIVED, "a section 2 cell persisted a column onto time_series"

print("section 2 self-check passed")
print(f"  {len(SETTING):,} delivery days, target hours partition the record, cutoff = issue time − {describe(DATA_INFO['actuals_lag'])}")
print(f"  capacity never used before its publication; {(~FORECASTABLE).sum()} unforecastable days")

---

## 3 Windows and split methods

- **Test window:** the last 365 **complete** delivery days, ending with the last day whose hours all
  carry the actual and every SMARD forecast. **Validation window:** the 365 delivery days before it.
  Both are derived from the data.
- **Training rows of a fit:** the delivery days inside the 24-month window ending at the fit's
  cutoff whose target hours are all observed by that cutoff (up to `DAY−2` for a fit issued on
  `DAY−1`). Unforecastable days and days with a missing actual are left out.
- **Tuning:** a rolling walk-forward over the validation year, once per grid configuration. The
  configuration with the lowest validation MAE is selected and **frozen**: both split methods reuse
  it unchanged, and refits never re-run the search.
- **Static:** one fit issued for the first test day. It forecasts the whole test year with daily
  updated inputs and frozen parameters.
- **Rolling:** the same first fit, then a refit every `refit_every` on the sliding 24-month window.
- Test-year fits train on windows that **include the validation year**: tuning only chose the
  configuration.

Both split methods forecast **identical test hours** with equal window lengths, so the only
difference between them is the refitting.

### 3.1 Test and validation windows

In [ ]:
JOINT_COLUMNS = ["residual_load", "fc_residual_load", "fc_grid_load", "fc_gen_wind_solar"]

by_day = pd.DataFrame(
    {
        "actual": time_series["residual_load"].notna(),
        "joint": time_series[JOINT_COLUMNS].notna().all(axis=1),
        "stamp": time_series.index,
    },
    index=time_series.index,
).groupby(DAY_OF_ROW)
DAY_INFO = pd.DataFrame(
    {
        "rows": by_day.size(),
        "first_row": by_day["stamp"].min(),
        "last_row": by_day["stamp"].max(),
        "actual_complete": by_day["actual"].all(),
        "jointly_observed": by_day["joint"].all(),
    }
)
assert DAY_INFO.index.equals(DAYS)

# A day covers its full clock span when it starts at 00:00, ends at the last hour before midnight and
# meets the completeness rule (the spring-DST day has 23 rows and passes).
full_span = (
    (DAY_INFO["first_row"] == DAY_INFO.index)
    & (DAY_INFO["last_row"] == DAY_INFO.index + pd.Timedelta(days=1) - RESOLUTION)
    & (DAY_INFO["rows"] >= MIN_OBS_PER_DAY)
)
TRAINABLE = FORECASTABLE & full_span & DAY_INFO["actual_complete"]
COMPLETE = full_span & DAY_INFO["jointly_observed"]

TEST_END = COMPLETE.index[COMPLETE][-1]
TEST_DAYS = DAYS[(DAYS > TEST_END - WINDOWS["test"]) & (DAYS <= TEST_END)]
VAL_DAYS = DAYS[(DAYS >= TEST_DAYS[0] - WINDOWS["validation"]) & (DAYS < TEST_DAYS[0])]


def hours_of(days):
    """The record's local hours of the delivery days in `days`."""
    return time_series.index[DAY_OF_ROW.isin(days)]


# Spec 04's trailing_365 window: 365 days of hours ending at the last jointly observed hour.
jointly = time_series[JOINT_COLUMNS].notna().all(axis=1)
JOINT_END = time_series.index[jointly][-1]
trailing_365 = time_series.index[(time_series.index > JOINT_END - WINDOWS["test"]) & (time_series.index <= JOINT_END)]

windows = pd.DataFrame(
    {
        name: {
            "first day": f"{days[0]:%Y-%m-%d}",
            "last day": f"{days[-1]:%Y-%m-%d}",
            "days": len(days),
            "hours": len(hours_of(days)),
            "unforecastable days": int((~FORECASTABLE.loc[days]).sum()),
            "days without a complete actual": int((~TRAINABLE.loc[days] & FORECASTABLE.loc[days]).sum()),
        }
        for name, days in [("validation", VAL_DAYS), ("test", TEST_DAYS)]
    }
).T
print("Test and validation windows (delivery days, local time)")
display(windows)

print(f"last jointly observed hour : {JOINT_END:%Y-%m-%d %H:%M}")
if hours_of(TEST_DAYS).equals(trailing_365):
    print(f"test window                : identical to spec 04's trailing_365 window ({len(trailing_365):,} h)")
else:
    print(
        f"test window                : ends {TEST_END:%Y-%m-%d}; the incomplete day after it is left out, "
        f"so it differs from spec 04's trailing_365 window ({len(trailing_365):,} h)"
    )

### 3.2 Fit schedule and training windows

One schedule per run. The two validation runs exist for tuning and for calibrating the prediction
bands (§6): the rolling band from the tuning walk-forward, the static band from one frozen fit over
the validation year.

In [ ]:
def fit_days_for(window_days, refit_every):
    """Issue days of the fits serving `window_days`: the first day, then one every `refit_every` (None: never)."""
    if refit_every is None:
        return window_days[:1]
    return pd.date_range(window_days[0], window_days[-1], freq=refit_every)


def training_days(fit_day):
    """Delivery days a fit issued for `fit_day` trains on (see §3's training-row rule)."""
    cutoff = SETTING.at[fit_day, "cutoff"]
    in_window = DAYS >= cutoff - WINDOWS["train"]
    observed_by_cutoff = available(DAY_INFO["last_row"], cutoff).to_numpy()
    return DAYS[in_window & observed_by_cutoff & TRAINABLE.to_numpy()]


RUNS = {
    "validation_rolling": {
        "split_method": "rolling", "days": VAL_DAYS, "refit_every": WINDOWS["refit_every"], "enabled": True,
        "purpose": "tuning; calibrates the rolling band",
    },
    "validation_static": {
        "split_method": "static", "days": VAL_DAYS, "refit_every": None, "enabled": TRAIN_TEST_SPLIT_METHOD["static"],
        "purpose": "calibrates the static band",
    },
    "test_static": {
        "split_method": "static", "days": TEST_DAYS, "refit_every": None, "enabled": TRAIN_TEST_SPLIT_METHOD["static"],
        "purpose": "test year, one frozen fit",
    },
    "test_rolling": {
        "split_method": "rolling", "days": TEST_DAYS, "refit_every": WINDOWS["refit_every"],
        "enabled": TRAIN_TEST_SPLIT_METHOD["rolling"],
        "purpose": f"test year, refit every {describe(WINDOWS['refit_every'])}",
    },
}
for run in RUNS.values():
    run["fit_days"] = fit_days_for(run["days"], run["refit_every"])

for name, run in RUNS.items():
    window_start = SETTING.at[run["fit_days"][0], "cutoff"] - WINDOWS["train"]
    if window_start < time_series.index[0]:
        raise ValueError(
            f"{name}: the first fit's {describe(WINDOWS['train'])} training window starts {window_start:%Y-%m-%d}, "
            f"before the record ({time_series.index[0]:%Y-%m-%d}). Shorten WINDOWS['train'] or re-fetch more history."
        )


def span(days):
    return f"{days[0]:%Y-%m-%d} .. {days[-1]:%Y-%m-%d} ({len(days)} days)"


schedule = pd.DataFrame(
    {
        name: {
            "split method": run["split_method"],
            "purpose": run["purpose"],
            "enabled": run["enabled"],
            "fits": len(run["fit_days"]),
            "forecast days": span(run["days"]),
            "first fit: training days": span(training_days(run["fit_days"][0])),
            "last fit: training days": span(training_days(run["fit_days"][-1])),
        }
        for name, run in RUNS.items()
    }
).T
print("Fit schedule per run")
display(schedule)

if RUNS["test_static"]["enabled"] and RUNS["test_rolling"]["enabled"]:
    rolling_fits = RUNS["test_rolling"]["fit_days"]
    shared_until = rolling_fits[1] - pd.Timedelta(days=1) if len(rolling_fits) > 1 else TEST_DAYS[-1]
    print(
        f"static and rolling share their first fit, so they forecast identically for "
        f"{TEST_DAYS[0]:%Y-%m-%d} .. {shared_until:%Y-%m-%d}: that stretch cannot differ between them."
    )

### 3.3 The split engine

Each model family supplies one function, registered in `FAMILY_FIT` in the Models section:
`fit(key, config, train_days, forecast_days)` returns `predict(day)`, which forecasts every target
hour of one delivery day. The engine calls it per scheduled fit and applies the failure rules:

- A fit that **raises** leaves its days empty until the next successful fit. The previous fit does
  not keep forecasting, and no other model fills in.
- A day whose forecast **raises** stays empty.
- A **convergence warning** is not a failure: the forecast is kept and the warning is counted.

`tune` runs the validation walk-forward once per grid configuration and selects the lowest MAE on
the hours all configurations forecast. `evaluate_model` then freezes that configuration and runs
every enabled split method.

In [ ]:
FAMILY_FIT = {}   # family -> fit function, filled in the Models section


def grid_configurations(model):
    """Every configuration of a registry entry's tuning grid, merged over its fixed parameters."""
    names = list(model["grid"])
    return [{**model["params"], **dict(zip(names, values))} for values in itertools.product(*model["grid"].values())]


def run_split(key, config, run):
    """Forecast the days of `run` with model `key` under `config`, each day by the latest fit at or before it.

    Returns the hourly forecast over the run's hours (empty where no forecast exists) and one log row per fit.
    """
    days, fit_days = RUNS[run]["days"], RUNS[run]["fit_days"]
    serving_fit = fit_days.searchsorted(days, side="right") - 1
    pieces, log = [], []

    for i, fit_day in enumerate(fit_days):
        serve = days[(serving_fit == i) & FORECASTABLE.loc[days].to_numpy()]
        train = training_days(fit_day)
        entry = {"fit_day": fit_day, "train_days": len(train), "forecast_days": len(serve), "status": "ok",
                 "failed_days": 0, "convergence_warnings": 0, "other_warnings": 0}

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            start = time.perf_counter()
            try:
                predict = FAMILY_FIT[MODELS[key]["family"]](key, config, train, serve)
            except Exception as error:
                predict, entry["status"] = None, f"failed: {type(error).__name__}: {error}"
            entry["fit_seconds"] = time.perf_counter() - start

            start = time.perf_counter()
            for day in serve if predict is not None else []:
                try:
                    forecast = predict(day)
                except Exception:
                    entry["failed_days"] += 1
                    continue
                assert forecast.index.equals(SETTING.at[day, "target_hours"]), f"{key}: {day:%Y-%m-%d} is not forecast hour by hour"
                pieces.append(forecast)
            entry["forecast_seconds"] = time.perf_counter() - start

        entry["convergence_warnings"] = sum(issubclass(w.category, ConvergenceWarning) for w in caught)
        entry["other_warnings"] = len(caught) - entry["convergence_warnings"]
        log.append(entry)

    forecast = pd.concat(pieces) if pieces else pd.Series(dtype=float)
    return forecast.reindex(hours_of(days)).astype(float), pd.DataFrame(log)


def tune(key):
    """Validation walk-forward for every grid configuration of `key`; selects the lowest MAE."""
    actual = time_series["residual_load"]
    trials = []
    for config in grid_configurations(MODELS[key]):
        forecast, log = run_split(key, config, "validation_rolling")
        # A configuration without a single forecast is a failure; it must not empty the others' hours.
        trials.append({"config": config, "forecast": forecast, "log": log, "usable": forecast.notna().any()})

    hours = actual.index[actual.notna()]
    for t in trials:
        if t["usable"]:
            hours = hours.intersection(t["forecast"].index[t["forecast"].notna()])

    rows = []
    for t in trials:
        grid_point = {name: t["config"][name] for name in MODELS[key]["grid"]}
        mae = (t["forecast"][hours] - actual[hours]).abs().mean() if t["usable"] else np.nan
        rows.append({**grid_point, "MAE": mae, "hour_count": len(hours) if t["usable"] else 0,
                     "failed fits": int((t["log"]["status"] != "ok").sum()),
                     "seconds": t["log"][["fit_seconds", "forecast_seconds"]].to_numpy().sum()})
    table = pd.DataFrame(rows)
    best = table["MAE"].idxmin() if table["MAE"].notna().any() else None
    return {
        "table": table,
        "selected": trials[best]["config"] if best is not None else None,
        "forecast": trials[best]["forecast"] if best is not None else None,
        "log": trials[best]["log"] if best is not None else None,
        "seconds": table["seconds"].sum(),
    }


def evaluate_model(key):
    """Tune `key` on the validation year, then run every enabled split method with the frozen configuration."""
    tuning = tune(key)
    result = {"tuning": tuning, "runs": {}}
    if tuning["selected"] is None:
        return result
    result["runs"]["validation_rolling"] = (tuning["forecast"], tuning["log"])
    for run in ["validation_static", "test_static", "test_rolling"]:
        if RUNS[run]["enabled"]:
            result["runs"][run] = run_split(key, tuning["selected"], run)
    return result

### 3.4 Self-check

In [ ]:
assert len(TEST_DAYS) == WINDOWS["test"].days and len(VAL_DAYS) == WINDOWS["validation"].days
assert VAL_DAYS[-1] + pd.Timedelta(days=1) == TEST_DAYS[0], "validation must end the day before the test window"
assert not VAL_DAYS.intersection(TEST_DAYS).size, "validation and test windows overlap"
assert COMPLETE.loc[TEST_END] and not COMPLETE.loc[DAYS > TEST_END].any(), "the test window must end on the last complete day"

for name, run in RUNS.items():
    for fit_day in run["fit_days"]:
        train = training_days(fit_day)
        assert available(DAY_INFO.loc[train, "last_row"], SETTING.at[fit_day, "cutoff"]).all()
        assert TRAINABLE.loc[train].all() and (train < fit_day).all()
        if name.startswith("validation"):
            assert (train < TEST_DAYS[0]).all(), f"{name}: tuning must never see a test day"

assert RUNS["test_static"]["fit_days"][0] == RUNS["test_rolling"]["fit_days"][0], "static and rolling share the first fit"
assert training_days(RUNS["test_static"]["fit_days"][0]).intersection(VAL_DAYS).size, "test fits must include the validation year"
assert list(time_series.columns) == SERIES + DERIVED, "a section 3 cell persisted a column onto time_series"

print("section 3 self-check passed")
print(f"  validation {VAL_DAYS[0]:%Y-%m-%d} .. {VAL_DAYS[-1]:%Y-%m-%d}, test {TEST_DAYS[0]:%Y-%m-%d} .. {TEST_DAYS[-1]:%Y-%m-%d}, no overlap")
print("  every fit trains only on days observed by its cutoff; no validation run trains on a test day")

---

## 4 Models

| Row | What it is | Inputs |
|---|---|---|
| `seasonal_naive` | the actual at the same local hour seven days earlier | `residual_load` |
| `sarimax_fourier` | a regression with ARIMA errors | SMARD's two component forecasts, Fourier terms for a day and a week, the holiday flag |
| `lgbm_direct`, `xgb_direct` | a booster predicts residual load | the enabled `FEATURES` groups |
| `lgbm_hybrid`, `xgb_hybrid` | OLS stage 1 plus a booster on its residuals | stage 1: SMARD's two component forecasts and a trend; stage 2: the enabled `FEATURES` groups |

`fc_residual_load` is an input of **no** model. It equals `fc_grid_load − fc_gen_wind_solar` exactly
(spec 04), so it would add nothing but collinearity. This section defines the models; §5 fits them.

### 4.1 Design rows for the boosters

One row per target hour, built **as at its own issue time** by `design_rows`. The groups are
**provisional**: the team's feature-engineering work may replace them.

| Group | Columns | Available because |
|---|---|---|
| calendar | local hour, day of week, month, weekend, holiday | always known |
| smard_forecast | `fc_grid_load`, `fc_gen_wind_solar` for the target hour | published by 18:00 on `DAY−1` |
| lags | `residual_load` at the same local hour on `DAY−2` and `DAY−7`; the last value before the cutoff | observed before the cutoff (§2.2) |
| recent_smard_error | mean `err_grid_load` and `err_renewables` over the 24 h ending at the cutoff | observed before the cutoff |
| capacity | `cap_wind_off + cap_wind_on + cap_solar` as published by the issue time | publication rule (§2.3) |

A lag whose source hour does not exist (local 02:00 on a spring DST day) stays empty. Both boosters
handle missing values natively, so nothing is filled. `design_rows` takes the data as arguments, so
the leakage test (§5) can rebuild the rows from data cut off at a day's cutoff.

In [ ]:
DE_HOLIDAYS = holidays.country_holidays("DE", years=YEARS)   # one source of truth, no subdiv


def is_holiday(index):
    """1.0 where the local calendar date of a timestamp is a German federal holiday, else 0.0."""
    return pd.Series(index.normalize().date, index=index).isin(set(DE_HOLIDAYS)).astype(float)


LAG_NAME = {lag: f"rl_lag_{lag / pd.Timedelta(hours=1):g}h" for lag in FEATURE_WINDOWS["same_hour_lags"]}
FEATURE_GROUPS = {
    "calendar": ["hour", "dow", "month", "is_weekend", "holiday"],
    "smard_forecast": ["fc_grid_load", "fc_gen_wind_solar"],
    "lags": [*LAG_NAME.values(), "rl_last_available"],
    "recent_smard_error": ["err_grid_load_recent", "err_renewables_recent"],
    "capacity": ["cap_total"],
}


def design_rows(days, frame=time_series, errors=smard_errors):
    """Booster design rows for every target hour of `days`, each built as at its own issue time.

    `frame` and `errors` default to the full data; the leakage test passes copies cut at a cutoff.
    """
    days = pd.DatetimeIndex(days).sort_values()
    hours = frame.index[frame.index.normalize().isin(days)]
    last_available = pd.DatetimeIndex(ROW_SETTING.loc[hours, "cutoff"] - RESOLUTION)
    actual = frame["residual_load"]
    recent = errors[["err_grid_load", "err_renewables"]].rolling(FEATURE_WINDOWS["recent_smard_error"]).mean()
    capacity = published_capacity(SETTING.loc[days, "issue_time"], capacity_table(frame)).set_axis(days)

    rows = pd.DataFrame(index=hours)
    rows["hour"] = hours.hour
    rows["dow"] = hours.dayofweek
    rows["month"] = hours.month
    rows["is_weekend"] = (hours.dayofweek >= 5).astype(float)
    rows["holiday"] = is_holiday(hours).to_numpy()
    rows["fc_grid_load"] = frame.loc[hours, "fc_grid_load"].to_numpy()
    rows["fc_gen_wind_solar"] = frame.loc[hours, "fc_gen_wind_solar"].to_numpy()
    for lag, name in LAG_NAME.items():
        rows[name] = actual.reindex(hours - lag).to_numpy()
    rows["rl_last_available"] = actual.reindex(last_available).to_numpy()
    rows["err_grid_load_recent"] = recent["err_grid_load"].reindex(last_available).to_numpy()
    rows["err_renewables_recent"] = recent["err_renewables"].reindex(last_available).to_numpy()
    rows["cap_total"] = capacity["cap_total"].reindex(hours.normalize()).to_numpy()
    return rows


DESIGN = design_rows(DAYS)

print("Booster feature groups (provisional)")
display(pd.DataFrame(
    {group: {"switch": "on" if FEATURES[group] else "off", "columns": ", ".join(columns)} for group, columns in FEATURE_GROUPS.items()}
).T)

missing = DESIGN.isna().sum()
print(f"\nDESIGN: {len(DESIGN):,} rows x {DESIGN.shape[1]} columns. Empty values over the whole record:")
print(missing[missing > 0].to_string() if missing.any() else "  none")

example_hours = SETTING.at[TEST_DAYS[0], "target_hours"][[0, len(SETTING.at[TEST_DAYS[0], "target_hours"]) // 2, -1]]
print(f"\nDesign rows of three target hours of the first test day ({TEST_DAYS[0]:%Y-%m-%d}; MWh, capacity in MW)")
display(DESIGN.loc[example_hours].T.rename(columns=lambda t: f"{t:%H:%M}"))

### 4.2 Seasonal naive `DAY−7`

The floor: the actual `residual_load` at the same local hour seven days earlier. It needs no fit and
is identical in both split methods, so it appears once, with `split_method = "none"`. If `DAY−7` is a
spring DST day, its local 02:00 does not exist and that forecast hour stays empty.

In [ ]:
NAIVE_LAG = FIXED["seasonal_naive"]["lag"]
if not available(time_series.index - NAIVE_LAG, ROW_SETTING["cutoff"].to_numpy()).all():
    raise ValueError(f"seasonal naive's lag of {describe(NAIVE_LAG)} reaches past the availability cutoff")


def seasonal_naive(days):
    """The actual residual load NAIVE_LAG earlier at the same local hour, for the forecastable days of `days`."""
    hours = hours_of(days)
    forecast = time_series["residual_load"].reindex(hours - NAIVE_LAG).set_axis(hours)
    return forecast.where(FORECASTABLE.reindex(hours.normalize()).to_numpy())


NAIVE = {"validation": seasonal_naive(VAL_DAYS), "test": seasonal_naive(TEST_DAYS)}

print("Seasonal naive per window")
display(pd.DataFrame(
    {name: {"hours": len(fc), "hours with a forecast": int(fc.notna().sum()), "hours without a source": int(fc.isna().sum())}
     for name, fc in NAIVE.items()}
).T)

### 4.3 `sarimax_fourier`

A regression with ARIMA errors, everything fixed in its registry entry:

- **Exogenous inputs:** the columns named in `exog` (`fc_grid_load`, `fc_gen_wind_solar`, the holiday
  flag) and Fourier terms for the periods and orders in `fourier` (1 day with `K = 4`, 1 week with
  `K = 3`), computed on the local clock.
- **Error model:** ARIMA with the fixed `order` `(1, 0, 1)` and an intercept, no order search. With
  `d = 0` the model reads as "regression on SMARD's forecasts plus seasons, with AR/MA errors that
  carry SMARD's recent error forward". With `d = 1` it would carry the level of the last actual
  forward over the 10–33 h to the target hours.
- **Fit** on the training days' rows. **Forecast:** the fitted parameters are applied once (`apply`,
  no re-estimation) to the series from the training window's start to the end of the fit's forecast
  stretch. Each day is then one dynamic prediction, starting at the first row not yet available (the
  row stamped 15:00 on `DAY−1`) and running to the end of `DAY`, with SMARD's forecasts as inputs.
  Only `DAY`'s hours are kept. A dynamic prediction uses only observations before its start, so no
  daily state updates are needed; the leakage test (§5) confirms it.
- **Series:** the local-time rows as they are, as evenly spaced steps: no UTC grid and no filling.
  The missing spring 02:00 is one skipped step per year; the autumn fold, which SMARD collapses into
  one row, needs nothing. The spacing is therefore off by one hour twice a year, which is negligible
  for hourly load. Rows without a SMARD forecast (an unforecastable day) are dropped the same way.

In [ ]:
def sarimax_exog(index, params):
    """SARIMAX's exogenous inputs for the rows `index`: the named columns plus Fourier terms on the local clock."""
    columns = {
        name: is_holiday(index).to_numpy() if name == "holiday" else time_series.loc[index, name].to_numpy()
        for name in params["exog"]
    }
    origin = pd.Timestamp(0)   # any fixed origin: sine and cosine together absorb the phase
    for period, order in params["fourier"].items():
        phase = np.asarray(((index - origin) % period) / period)
        label = f"{period / pd.Timedelta(hours=1):g}h"
        for k in range(1, order + 1):
            columns[f"sin_{label}_{k}"] = np.sin(2 * np.pi * k * phase)
            columns[f"cos_{label}_{k}"] = np.cos(2 * np.pi * k * phase)
    return pd.DataFrame(columns, index=index)


def sarimax_series(first_row, last_row, params):
    """The rows from `first_row` to `last_row` that carry every exogenous input, and those inputs."""
    rows = time_series.index[(time_series.index >= first_row) & (time_series.index <= last_row)]
    exog = sarimax_exog(rows, params)
    keep = exog.notna().all(axis=1).to_numpy()
    return rows[keep], exog[keep]


def fit_sarimax(key, config, train_days, forecast_days):
    """Fit on the training days; forecast each day by one dynamic prediction from its first unavailable row."""
    train = hours_of(train_days)
    fitted = SARIMAX(
        time_series.loc[train, "residual_load"].to_numpy(),
        exog=sarimax_exog(train, config).to_numpy(),
        order=config["order"],
        trend=config["trend"],
    ).fit(disp=False)
    if not len(forecast_days):
        return lambda day: None

    rows, exog = sarimax_series(train[0], hours_of(forecast_days)[-1], config)
    applied = fitted.apply(endog=time_series.loc[rows, "residual_load"].to_numpy(), exog=exog.to_numpy())

    def predict(day):
        hours = SETTING.at[day, "target_hours"]
        start = rows.searchsorted(SETTING.at[day, "cutoff"] - RESOLUTION, side="right")  # first unavailable row
        end = rows.get_loc(hours[-1])
        mean = applied.get_prediction(start=start, end=end, dynamic=True).predicted_mean
        return pd.Series(mean, index=rows[start:end + 1])[hours]

    return predict


# What the default windows do to SARIMAX's series: rows dropped for a missing forecast, DST steps skipped.
SARIMAX_PARAMS = MODELS["sarimax_fourier"]["params"]
series_start = min(training_days(run["fit_days"][0])[0] for run in RUNS.values())
span_rows = time_series.index[(time_series.index >= series_start) & (time_series.index <= hours_of(TEST_DAYS)[-1])]
kept_rows, _ = sarimax_series(span_rows[0], span_rows[-1], SARIMAX_PARAMS)
print(f"SARIMAX series in the default windows: {series_start:%Y-%m-%d} .. {TEST_END:%Y-%m-%d}, {len(span_rows):,} rows")
print(f"  rows dropped for a missing SMARD forecast : {len(span_rows) - len(kept_rows)}")
print(f"  spring-DST hours skipped as a single step : {int(time_series.loc[span_rows, 'spans_gap'].sum())}")
print(f"  exogenous inputs                          : {sarimax_exog(span_rows[:1], SARIMAX_PARAMS).shape[1]} plus the intercept")

### 4.4 Stationarity check (KPSS)

A **check, not a decision**: one KPSS test on the residuals of an OLS fit of residual load on
`sarimax_fourier`'s exogenous inputs, over the first validation fit's training window (the
`WINDOWS["train"]` before the validation start). `d` stays at 0 whatever the test says; if KPSS rejects
stationarity, the team can change `d` in the registry.

- Differencing, if ever needed, happens **inside** ARIMA through `d`, never on the series outside the
  model.
- The boosters need no stationarity, but they **cannot extrapolate** beyond the target range they
  were trained on.
- Box-Cox or log transforms are ruled out: residual load is negative in some hours.

In [ ]:
kpss_rows = hours_of(training_days(RUNS["validation_rolling"]["fit_days"][0]))
ols = sm.OLS(
    time_series.loc[kpss_rows, "residual_load"].to_numpy(),
    sm.add_constant(sarimax_exog(kpss_rows, SARIMAX_PARAMS).to_numpy()),
).fit()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(ols.resid, regression="c", nlags="auto")
p_bounded = any(w.category.__name__ == "InterpolationWarning" for w in caught)

print(f"KPSS on OLS residuals, {kpss_rows[0]:%Y-%m-%d} .. {kpss_rows[-1]:%Y-%m-%d} ({len(kpss_rows):,} h)")
print(f"  statistic {kpss_stat:.3f} with {kpss_lags} lags; 5 % critical value {kpss_crit['5%']}")
print(f"  p-value {'outside the table (reported as the bound) ' if p_bounded else ''}{kpss_p:.3f}")
print(
    f"  -> {'rejects' if kpss_p < 0.05 else 'does not reject'} stationarity at 5 %. "
    f"d stays {SARIMAX_PARAMS['order'][1]} (registry); this is a check, not a decision."
)

### 4.5 Boosters: direct and hybrid

LightGBM and XGBoost regressors with the fixed seed `SEED`, each in two architectures:

- **Direct:** the booster predicts `residual_load` from the enabled feature groups.
- **Hybrid:** stage 1 is an OLS model on `fc_grid_load`, `fc_gen_wind_solar` and a linear trend
  (days since the first training day). Stage 2 is the booster, fitted on stage 1's in-sample
  residuals with the enabled feature groups. The forecast is stage 1 + stage 2. **Stage 1's inputs are
  fixed**: the `FEATURES` switches affect only the direct booster and stage 2.
- **No early stopping.** `n_estimators` is a grid value; early stopping would need a third
  time-ordered split inside each training window.

**Expected difference:** the hybrid can extrapolate the level through stage 1, while the direct
booster is capped at the range of its training targets. This matters in the low tail, where the test
year may reach more negative residual load than the training window; the extremes scoreboard (§7)
prints both minima.

In [ ]:
STAGE1_INPUTS = ["fc_grid_load", "fc_gen_wind_solar"]   # fixed: independent of the FEATURES switches
BOOSTER = {"lightgbm": lgb.LGBMRegressor, "xgboost": xgb.XGBRegressor}


def feature_columns():
    """The design-row columns of every enabled FEATURES group."""
    return [column for group, enabled in FEATURES.items() if enabled for column in FEATURE_GROUPS[group]]


def stage1_design(rows, origin):
    """Hybrid stage 1 inputs: an intercept, the two SMARD forecasts and a linear trend in days since `origin`."""
    trend = np.asarray((rows.index - origin) / pd.Timedelta(days=1))
    return np.column_stack([np.ones(len(rows)), rows[STAGE1_INPUTS].to_numpy(), trend])


def fit_booster(key, config, train_days, forecast_days):
    """Direct: the booster predicts residual load. Hybrid: OLS stage 1 plus a booster on its residuals."""
    columns = feature_columns()
    if not columns:
        raise ValueError("every FEATURES group is switched off")
    train = DESIGN.loc[hours_of(train_days)]
    target = time_series.loc[train.index, "residual_load"].to_numpy()

    hybrid = MODELS[key]["architecture"] == "hybrid"
    if hybrid:
        origin = train_days[0]
        coef, *_ = np.linalg.lstsq(stage1_design(train, origin), target, rcond=None)
        target = target - stage1_design(train, origin) @ coef
    booster = BOOSTER[MODELS[key]["family"]](**config).fit(train[columns], target)
    if not len(forecast_days):
        return lambda day: None

    future = DESIGN.loc[hours_of(forecast_days)]
    level = stage1_design(future, origin) @ coef if hybrid else 0.0
    forecast = pd.Series(level + booster.predict(future[columns]), index=future.index)
    return lambda day: forecast[SETTING.at[day, "target_hours"]]


FAMILY_FIT.update({"sarimax": fit_sarimax, "lightgbm": fit_booster, "xgboost": fit_booster})

print(f"booster features in use ({len(feature_columns())}): {', '.join(feature_columns())}")
print(f"hybrid stage 1 inputs (fixed): intercept, {', '.join(STAGE1_INPUTS)}, trend in days")
print(f"FAMILY_FIT: {', '.join(f'{family} -> {fn.__name__}' for family, fn in FAMILY_FIT.items())}")

### 4.6 Self-check

In [ ]:
group_columns = [column for columns in FEATURE_GROUPS.values() for column in columns]
assert list(DESIGN.columns) == group_columns and DESIGN.index.equals(time_series.index)
assert set(feature_columns()) <= set(group_columns)

model_inputs = set(group_columns) | set(STAGE1_INPUTS)
for key, m in MODELS.items():
    if m["family"] == "sarimax":
        model_inputs |= set(sarimax_exog(time_series.index[:1], m["params"]).columns)
assert "fc_residual_load" not in model_inputs, "fc_residual_load must never be a model input"

assert {m["family"] for m in MODELS.values()} <= set(FAMILY_FIT), "a registry family has no fit function"
assert NAIVE["test"].index.equals(hours_of(TEST_DAYS)) and NAIVE["validation"].index.equals(hours_of(VAL_DAYS))
assert DESIGN.loc[hours_of(TEST_DAYS), "cap_total"].notna().all(), "every test hour has a published capacity value"
assert list(time_series.columns) == SERIES + DERIVED, "a section 4 cell persisted a column onto time_series"

print("section 4 self-check passed")
print(f"  {DESIGN.shape[1]} design columns in {len(FEATURE_GROUPS)} groups; fc_residual_load is no model's input")
print(f"  fit functions registered for {sorted(FAMILY_FIT)}")

---

## 5 Fitting and leakage test

Every enabled registry model is tuned on the validation year and then run under every enabled split
method on the test year, with the frozen configuration (`evaluate_model`, §3.3). Seasonal naive needs
no fit (§4.2). With the default registry this takes a few minutes; the spec's budget is about an hour.

### 5.1 Fit every enabled model

In [ ]:
RESULTS = {}
fitting_started = time.perf_counter()
for key, model in MODELS.items():
    if not model["enabled"]:
        print(f"{key:<16} switched off in the registry")
        continue
    started = time.perf_counter()
    RESULTS[key] = evaluate_model(key)
    print(f"{key:<16} tuned and run in {time.perf_counter() - started:,.0f} s")

FITTING_SECONDS = time.perf_counter() - fitting_started
print(f"\nall enabled models: {FITTING_SECONDS / 60:,.1f} min")
if FITTING_SECONDS > 3600:
    print("-> clearly over the spec's one-hour budget: raise it with the team rather than shrinking windows or grids.")

### 5.2 Selected configurations (frozen)

The tuning walk-forward per grid configuration, scored by residual-load MAE on the validation hours
that every configuration of the model forecast. The selected configuration is reused unchanged by
both split methods; refits never re-run the search.

In [ ]:
tuning_rows = []
for key, result in RESULTS.items():
    grid, selected = list(MODELS[key]["grid"]), result["tuning"]["selected"]
    for _, row in result["tuning"]["table"].iterrows():
        tuning_rows.append({
            "model": key,
            "configuration": ", ".join(f"{name}={row[name]:g}" for name in grid) if grid else "fixed (no grid)",
            "validation MAE (MWh)": row["MAE"],
            "hour_count": row["hour_count"],
            "failed fits": row["failed fits"],
            "seconds": row["seconds"],
            "selected": selected is not None and all(row[name] == selected[name] for name in grid),
        })
TUNING = pd.DataFrame(tuning_rows).set_index(["model", "configuration"])

print("Tuning on the validation year (walk-forward, refit every " f"{describe(WINDOWS['refit_every'])})")
display(TUNING.style.format({"validation MAE (MWh)": "{:,.0f}", "hour_count": "{:,.0f}", "failed fits": "{:,.0f}", "seconds": "{:,.1f}"}))

SELECTED = {key: result["tuning"]["selected"] for key, result in RESULTS.items()}
print("Frozen for both split methods:")
for key, config in SELECTED.items():
    shown = {name: config[name] for name in MODELS[key]["grid"]} if config else None
    print(f"  {key:<16}: {shown if shown else ('fixed registry parameters' if config else 'NONE: every configuration failed')}")

### 5.3 Runtime, failures and warnings

Fit time counts the fits of each run; the test-year fits are what the accuracy scoreboard reports.
Tuning time is the grid walk-forwards plus the static validation run. A failed fit or day stays
empty (§3.3); a convergence warning keeps its forecast. Other warnings raised inside a fit are
counted here rather than printed.

In [ ]:
FIT_LOG = pd.concat(
    [log.assign(model=key, run=run) for key, result in RESULTS.items() for run, (_, log) in result["runs"].items()],
    ignore_index=True,
)

report = FIT_LOG.groupby(["model", "run"], sort=False).agg(
    fits=("fit_day", "size"),
    failed_fits=("status", lambda s: int((s != "ok").sum())),
    failed_days=("failed_days", "sum"),
    convergence_warnings=("convergence_warnings", "sum"),
    other_warnings=("other_warnings", "sum"),
    fit_seconds=("fit_seconds", "sum"),
    forecast_seconds=("forecast_seconds", "sum"),
)
print("Per model and run")
display(report.style.format({"fit_seconds": "{:,.1f}", "forecast_seconds": "{:,.1f}"}))

def run_seconds(key, run, columns=("fit_seconds",)):
    """Seconds spent in `columns` over every fit of one model's run."""
    rows = (FIT_LOG["model"] == key) & (FIT_LOG["run"] == run)
    return float(FIT_LOG.loc[rows, list(columns)].to_numpy().sum())


BOTH = ("fit_seconds", "forecast_seconds")
runtime_rows = {}
for key, result in RESULTS.items():
    row = {"tuning (s)": result["tuning"]["seconds"] + run_seconds(key, "validation_static", BOTH)}
    for run in ["test_static", "test_rolling"]:
        if run in result["runs"]:
            row[f"{RUNS[run]['split_method']}: test-year fits (s)"] = run_seconds(key, run)
    row["total incl. forecasting (s)"] = row["tuning (s)"] + sum(
        run_seconds(key, run, BOTH) for run in ["test_static", "test_rolling"] if run in result["runs"]
    )
    runtime_rows[key] = row
RUNTIME = pd.DataFrame(runtime_rows).T
print("Runtime per model")
display(RUNTIME.style.format("{:,.1f}"))

failures = FIT_LOG[FIT_LOG["status"] != "ok"]
print(f"failed fits: {len(failures)}")
if not failures.empty:
    display(failures[["model", "run", "fit_day", "status"]])

### 5.4 Leakage test by truncation

For a fixed-seed sample of delivery days from the training, validation and test windows, plus the
two days where leakage is easiest to get wrong (the latest 1 January and the latest spring-DST day),
the data is **cut off at that day's availability cutoff**: actuals and SMARD's errors up to the
cutoff, SMARD's forecasts up to the end of `DAY`.

- The day's design rows are rebuilt from the cut data and must be **identical** to the rows the
  models used. Any feature that peeks past the cutoff changes when the future is removed, so this
  catches leakage without tracking a timestamp per feature.
- For the sampled validation and test days, `sarimax_fourier`'s forecast from the fitting run must
  equal a dynamic prediction from the same fit applied to the series cut at the cutoff.
- Three fitting rules are asserted: linear stages and tuning never see a test row, the test year is
  forecast but never used for selection, and every split is time-ordered (no shuffling anywhere).

In [ ]:
LEAKAGE_SAMPLE_DAYS = 20        # drawn with SEED, spread evenly over the three windows
SARIMAX_TOLERANCE = 1e-6        # MWh: floating-point identical
MEASURED_COLUMNS = ["wind_off", "wind_on", "solar", "grid_load", "residual_load", "renewables", *CAP_COLUMNS]


def cut_at_cutoff(day):
    """The data as known at `day`'s issue time: measured values up to the cutoff, SMARD forecasts to the end of `day`."""
    cutoff, end = SETTING.at[day, "cutoff"], SETTING.at[day, "target_hours"][-1]
    frame, errors = time_series.loc[:end].copy(), smard_errors.loc[:end].copy()
    frame.loc[~available(frame.index, cutoff), MEASURED_COLUMNS] = np.nan
    errors.loc[~available(errors.index, cutoff), :] = np.nan   # every error needs the actual
    return frame, errors


pools = {
    "training": training_days(RUNS["validation_rolling"]["fit_days"][0]),
    "validation": VAL_DAYS[FORECASTABLE.loc[VAL_DAYS].to_numpy()],
    "test": TEST_DAYS[FORECASTABLE.loc[TEST_DAYS].to_numpy()],
}
rng = np.random.default_rng(SEED)
per_pool = np.diff(np.linspace(0, LEAKAGE_SAMPLE_DAYS, len(pools) + 1).round().astype(int))
SAMPLE = {
    name: pd.DatetimeIndex(sorted(rng.choice(days, size=n, replace=False)))
    for (name, days), n in zip(pools.items(), per_pool)
}
for edge in [new_year, dst_days[-1]]:
    for name, days in pools.items():
        if edge in days and edge not in SAMPLE[name]:
            SAMPLE[name] = SAMPLE[name].append(pd.DatetimeIndex([edge])).sort_values()

# 1. Design rows rebuilt from the cut data
design_checks = []
for window, days in SAMPLE.items():
    for day in days:
        frame, errors = cut_at_cutoff(day)
        rebuilt = design_rows([day], frame, errors)
        design_checks.append({"window": window, "day": day, "identical": rebuilt.equals(DESIGN.loc[rebuilt.index])})
design_checks = pd.DataFrame(design_checks)

# 2. SARIMAX: the fitting run's forecast against the same fit applied to the cut series
sarimax_checks = []
if RESULTS.get("sarimax_fourier", {}).get("runs"):
    config = SELECTED["sarimax_fourier"]
    fits = {}
    for window, candidates in [("validation", ["validation_static", "validation_rolling"]), ("test", ["test_static", "test_rolling"])]:
        run = next(r for r in candidates if r in RESULTS["sarimax_fourier"]["runs"])
        production = RESULTS["sarimax_fourier"]["runs"][run][0]
        fit_days = RUNS[run]["fit_days"]
        for day in SAMPLE[window]:
            fit_day = fit_days[fit_days.searchsorted(day, side="right") - 1]
            if fit_day not in fits:
                train = hours_of(training_days(fit_day))
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    fits[fit_day] = (train[0], SARIMAX(
                        time_series.loc[train, "residual_load"].to_numpy(),
                        exog=sarimax_exog(train, config).to_numpy(), order=config["order"], trend=config["trend"],
                    ).fit(disp=False))
            first_row, fitted = fits[fit_day]
            hours, cutoff = SETTING.at[day, "target_hours"], SETTING.at[day, "cutoff"]
            rows, exog = sarimax_series(first_row, hours[-1], config)
            endog = time_series.loc[rows, "residual_load"].where(available(rows, cutoff)).to_numpy()
            applied = fitted.apply(endog=endog, exog=exog.to_numpy())
            start, end = rows.searchsorted(cutoff - RESOLUTION, side="right"), rows.get_loc(hours[-1])
            cut = pd.Series(applied.get_prediction(start=start, end=end, dynamic=True).predicted_mean, index=rows[start:end + 1])
            sarimax_checks.append({"window": window, "day": day, "run": run,
                                   "max_abs_difference": float((cut[hours] - production[hours]).abs().max())})
sarimax_checks = pd.DataFrame(sarimax_checks)

# 3. Fitting rules
for key, result in RESULTS.items():
    tuning = result["tuning"]
    assert tuning["forecast"].index.equals(hours_of(VAL_DAYS)), f"{key}: selection saw hours outside the validation year"
    assert (tuning["table"]["hour_count"] <= len(hours_of(VAL_DAYS))).all()
    for run, (forecast, log) in result["runs"].items():
        assert forecast.index.equals(hours_of(RUNS[run]["days"]))
        assert log["fit_day"].is_monotonic_increasing and log["fit_day"].is_unique, f"{key}/{run}: fits out of time order"
        for fit_day in log["fit_day"]:
            train = training_days(fit_day)
            assert train.is_monotonic_increasing and (train < fit_day).all(), f"{key}/{run}: training after the fit day"
            if run.startswith("validation"):
                assert (train < TEST_DAYS[0]).all(), f"{key}/{run}: a tuning or calibration fit saw a test day"
assert kpss_rows[-1] < VAL_DAYS[0], "the KPSS OLS must not see validation or test rows"

assert design_checks["identical"].all(), f"design rows changed under truncation: {design_checks.loc[~design_checks['identical'], 'day'].tolist()}"
assert sarimax_checks.empty or (sarimax_checks["max_abs_difference"] <= SARIMAX_TOLERANCE).all(), sarimax_checks
LEAKAGE_PASSED = True

summary = design_checks.groupby("window", sort=False).agg(days=("day", "size"), identical_design_rows=("identical", "sum"))
if not sarimax_checks.empty:
    summary = summary.join(sarimax_checks.groupby("window").agg(
        sarimax_days=("day", "size"), sarimax_max_abs_difference_MWh=("max_abs_difference", "max")))
print(f"Leakage test by truncation: {len(design_checks)} delivery days (seed {SEED}, incl. {new_year:%Y-%m-%d} and {dst_days[-1]:%Y-%m-%d})")
display(summary)
print("passed: no rebuilt feature changed, SARIMAX ignores everything after the cutoff, all fitting rules hold")

### 5.5 Self-check

In [ ]:
enabled = [key for key, m in MODELS.items() if m["enabled"]]
assert list(RESULTS) == enabled, "every enabled registry model has a result"

# Static and rolling share their first fit, so their first stretch of the test year must be identical.
rolling_fits = RUNS["test_rolling"]["fit_days"]
shared_until = rolling_fits[1] if len(rolling_fits) > 1 else TEST_DAYS[-1] + pd.Timedelta(days=1)
first_stretch = hours_of(TEST_DAYS[TEST_DAYS < shared_until])
for key, result in RESULTS.items():
    runs = result["runs"]
    if "test_static" in runs and "test_rolling" in runs:
        assert runs["test_static"][0][first_stretch].equals(runs["test_rolling"][0][first_stretch]), f"{key}: shared first fit differs"

assert LEAKAGE_PASSED
assert list(time_series.columns) == SERIES + DERIVED, "a section 5 cell persisted a column onto time_series"

print("section 5 self-check passed")
print(f"  {len(RESULTS)} models fitted, configurations frozen: {', '.join(RESULTS)}")
print(f"  static and rolling identical over {len(first_stretch):,} shared hours; leakage test passed")

---

## 6 Prediction intervals

**One empirical method for every row** (split-conformal style), seasonal naive included:

- **Residual sign:** the band uses `r = actual − forecast`, the **opposite** sign to the project's
  `error = forecast − actual`. A positive `r` means the actual came out above the forecast.
- **Calibration, per split method,** the way the method is used:
  - rolling: the residuals of the tuning walk-forward (§5.2) of the selected configuration
  - static: the residuals of one frozen fit at the start of the validation year, forecasting the
    whole validation year without refitting. A model frozen for a year makes larger errors than a
    refitted one, so rolling residuals would make the static band too narrow.
  - seasonal naive: its validation-year residuals, for its single row
- **Quantiles:** the `(1 − level) / 2` and `(1 + level) / 2` quantiles of `r` **per local hour of
  day**, because the error size varies strongly with the hour.
- **Band:** `forecast + [q_low(r), q_high(r)]` for each test hour, with the quantiles of that hour.

The band is calibrated on the validation year and **not updated** during the test year, so drift
shows up as lost coverage (scoreboard C, §7). If a model is strongly biased at some hour, both
quantiles share a sign and the band does not contain the forecast itself; that is correct, not an
error. SMARD is a point forecast and gets no band.

In [ ]:
LOW_Q, HIGH_Q = (1 - INTERVAL["level"]) / 2, (1 + INTERVAL["level"]) / 2
CALIBRATED_ON = {"test_static": "validation_static", "test_rolling": "validation_rolling"}


def band_quantiles(forecast):
    """Per local hour of day, the LOW_Q and HIGH_Q quantiles of r = actual − forecast over the forecast's hours."""
    r = (time_series.loc[forecast.index, "residual_load"] - forecast).dropna()
    return r.groupby(r.index.hour).quantile([LOW_Q, HIGH_Q]).unstack().rename(columns={LOW_Q: "q_low", HIGH_Q: "q_high"})


def with_band(forecast, quantiles):
    """The forecast with its band: forecast + the calibrated quantiles of each hour's local hour of day."""
    q = quantiles.reindex(forecast.index.hour)
    return pd.DataFrame(
        {"forecast": forecast, "lower": forecast + q["q_low"].to_numpy(), "upper": forecast + q["q_high"].to_numpy()},
        index=forecast.index,
    )


# One entry per scoreboard row with a band: (model, split_method) -> test-hour forecast, lower, upper.
FORECASTS, CALIBRATION = {}, {}
for key, result in RESULTS.items():
    for run, calibration_run in CALIBRATED_ON.items():
        if run in result["runs"] and calibration_run in result["runs"]:
            row = (key, RUNS[run]["split_method"])
            CALIBRATION[row] = (calibration_run, band_quantiles(result["runs"][calibration_run][0]))
            FORECASTS[row] = with_band(result["runs"][run][0], CALIBRATION[row][1])
CALIBRATION[("seasonal_naive", "none")] = ("validation year", band_quantiles(NAIVE["validation"]))
FORECASTS[("seasonal_naive", "none")] = with_band(NAIVE["test"], CALIBRATION[("seasonal_naive", "none")][1])


def calibration_summary(row):
    calibrated_on, q = CALIBRATION[row]
    width = q["q_high"] - q["q_low"]
    return {
        "calibrated on": calibrated_on,
        "local hours": len(q),
        "mean width (MWh)": width.mean(),
        "narrowest hour": f"{width.idxmin():02d}:00 ({width.min():,.0f} MWh)",
        "widest hour": f"{width.idxmax():02d}:00 ({width.max():,.0f} MWh)",
        "hours excluding the forecast": int(((q["q_low"] > 0) | (q["q_high"] < 0)).sum()),
    }


summary = pd.DataFrame({row: calibration_summary(row) for row in CALIBRATION}).T.rename_axis(["model", "split_method"])
print(f"{INTERVAL['level']:.0%} bands: quantiles {LOW_Q:.3f} / {HIGH_Q:.3f} of r = actual − forecast per local hour, "
      "calibrated on the validation year")
display(summary.style.format({"mean width (MWh)": "{:,.0f}"}))

### 6.1 Self-check

In [ ]:
val_hours = hours_of(VAL_DAYS)
for row, (calibrated_on, q) in CALIBRATION.items():
    assert q.index.isin(range(24)).all() and (q["q_low"] <= q["q_high"]).all(), row
for key, result in RESULTS.items():
    for run, calibration_run in CALIBRATED_ON.items():
        if calibration_run in result["runs"]:
            assert result["runs"][calibration_run][0].index.equals(val_hours), f"{key}: a band saw hours outside the validation year"
assert NAIVE["validation"].index.equals(val_hours)

for row, frame in FORECASTS.items():
    assert frame.index.equals(hours_of(TEST_DAYS)), row
    banded = frame["forecast"].notna()
    assert (frame.loc[banded, "lower"] <= frame.loc[banded, "upper"]).all(), f"{row}: band edges crossed"
assert not any(row[0] == "smard" for row in FORECASTS), "SMARD is a point forecast and gets no band"
assert list(time_series.columns) == SERIES + DERIVED, "a section 6 cell persisted a column onto time_series"

print("section 6 self-check passed")
print(f"  {len(FORECASTS)} banded rows: {', '.join(f'{m} / {s}' for m, s in FORECASTS)}")
print("  every band calibrated on validation hours only, per local hour; SMARD has none")

---

## 7 Evaluation and scoreboard

**Rows:** every enabled registry model × split method with at least one forecast, seasonal naive
(`split_method = "none"`) and SMARD (`smard`, `"none"`).

**Common hours:** every row is scored on the same hours: the test hours where the actual, SMARD's
forecast, seasonal naive's forecast and every registry row's forecast all exist (a strict
intersection). SMARD is re-scored from `smard_forecast_errors_hourly.csv` on these hours, and a
cross-check confirms the file agrees with `data/smard.csv`. A row with no forecast at all (a failed
static fit) gets no scoreboard row and is left out of the intersection, so it cannot empty everyone
else's evaluation set. If one model breaks badly, switch it off in the registry and re-run.

**Metrics** follow spec 04:

- `error = forecast − actual`; **MAE**, **RMSE** (recomputed from the hourly errors, never averaged)
  and **bias**, with `hour_count` next to every value; no MAPE
- `skill = 1 − MAE_model / MAE_SMARD` on the same hours, in `%`; **positive = better than SMARD**
- **months beating SMARD** as `k of n`: `n` calendar months lie entirely inside the test window, and
  in `k` of them the row's MAE is below SMARD's. This is the robustness view in place of a
  significance test.

### 7.1 Common hours

In [ ]:
actual = time_series["residual_load"]
test_hours = hours_of(TEST_DAYS)
SMARD_ROW = ("smard", "none")
TEST_RUN = {RUNS[run]["split_method"]: run for run in ["test_static", "test_rolling"]}

ROWS = {row: frame["forecast"] for row, frame in FORECASTS.items() if frame["forecast"].notna().any()}
ROWS[SMARD_ROW] = smard_errors.loc[test_hours, "fc_residual_load"]
no_forecast = [row for row in FORECASTS if row not in ROWS]

has_actual = actual[test_hours].notna()
forecast_exists = pd.DataFrame({row: forecast.notna() for row, forecast in ROWS.items()}, index=test_hours)
COMMON = test_hours[(forecast_exists.all(axis=1) & has_actual).to_numpy()]

# SMARD re-scored from its hourly errors file, cross-checked against data/smard.csv on the common hours.
smard_error = smard_errors.loc[COMMON, "err_residual_load"]
SMARD_CROSS_CHECK = max(
    float((smard_error - (time_series.loc[COMMON, "fc_residual_load"] - actual[COMMON])).abs().max()),
    float((smard_errors.loc[COMMON, "residual_load"] - actual[COMMON]).abs().max()),
)
assert SMARD_CROSS_CHECK <= 1e-6, f"smard_forecast_errors_hourly.csv disagrees with data/smard.csv by {SMARD_CROSS_CHECK} MWh"

ERRORS = {row: forecast[COMMON] - actual[COMMON] for row, forecast in ROWS.items() if row != SMARD_ROW}
ERRORS[SMARD_ROW] = smard_error


def test_log(row):
    """The fit log of a registry row's test-year run."""
    return FIT_LOG[(FIT_LOG["model"] == row[0]) & (FIT_LOG["run"] == TEST_RUN[row[1]])]


losses = {}
for row, forecast in ROWS.items():
    entry = {"test hours with a forecast": int(forecast.notna().sum()),
             "test hours without a forecast": int((forecast.isna() & has_actual).sum())}
    if row[0] in MODELS:
        log = test_log(row)
        entry.update({"failed fits": int((log["status"] != "ok").sum()), "failed days": int(log["failed_days"].sum()),
                      "convergence warnings": int(log["convergence_warnings"].sum())})
    losses[row] = entry
print(f"test hours {len(test_hours):,}; with an actual {int(has_actual.sum()):,}; "
      f"common hours {len(COMMON):,} (lost {len(test_hours) - len(COMMON):,})")
print(f"left out, no forecast at all: {no_forecast if no_forecast else 'none'}")
print(f"SMARD cross-check (errors file vs data/smard.csv): max |difference| {SMARD_CROSS_CHECK:.1e} MWh")
display(pd.DataFrame(losses).T.rename_axis(["model", "split_method"]).style.format("{:,.0f}", na_rep="—"))

### 7.2 Scoreboard A: accuracy

Sorted by MAE. Fit time sums the split method's **test-year fits only**; tuning time is listed below
the table. In the SMARD row, skill, months beating SMARD and fit time are "—": they compare a row
against SMARD, and SMARD has no fit. Seasonal naive has no fit either.

In [ ]:
FULL_MONTHS = [
    month for month in TEST_DAYS.to_period("M").unique()
    if month.start_time in TEST_DAYS and month.end_time.normalize() in TEST_DAYS
]
month_of = COMMON.to_period("M")
monthly_mae = {row: e.abs().groupby(month_of).mean() for row, e in ERRORS.items()}
MAE_SMARD = ERRORS[SMARD_ROW].abs().mean()

ACCURACY_RECORDS, accuracy = [], {}
for row, e in ERRORS.items():
    entry = {"MAE": e.abs().mean(), "RMSE": np.sqrt((e ** 2).mean()), "bias": e.mean(), "hour_count": len(e),
             "skill vs SMARD (%)": np.nan, "months beating SMARD": np.nan, "fit time (s)": np.nan}
    for metric in ["MAE", "RMSE", "bias"]:
        ACCURACY_RECORDS.append((*row, "accuracy", metric, entry[metric], len(e)))
    if row != SMARD_ROW:
        wins = sum(monthly_mae[row][month] < monthly_mae[SMARD_ROW][month] for month in FULL_MONTHS)
        entry["skill vs SMARD (%)"] = 100 * (1 - entry["MAE"] / MAE_SMARD)
        entry["months beating SMARD"] = f"{wins} of {len(FULL_MONTHS)}"
        ACCURACY_RECORDS.append((*row, "accuracy", "skill_pct", entry["skill vs SMARD (%)"], len(e)))
        ACCURACY_RECORDS.append((*row, "accuracy", "months_beating_smard", wins, len(FULL_MONTHS)))
    if row[0] in MODELS:
        log = test_log(row)
        entry["fit time (s)"] = log["fit_seconds"].sum()
        ACCURACY_RECORDS.append((*row, "accuracy", "fit_seconds", entry["fit time (s)"], len(log)))
    accuracy[row] = entry

SCOREBOARD_A = pd.DataFrame(accuracy).T.rename_axis(["model", "split_method"]).sort_values("MAE")
print(f"Scoreboard A: accuracy on {len(COMMON):,} common test hours ({TEST_DAYS[0]:%Y-%m-%d} .. {TEST_DAYS[-1]:%Y-%m-%d}); "
      f"MAE, RMSE and bias in MWh; {len(FULL_MONTHS)} full calendar months")
display(SCOREBOARD_A.style.format(
    {"MAE": "{:,.0f}", "RMSE": "{:,.0f}", "bias": "{:+,.0f}", "hour_count": "{:,.0f}",
     "skill vs SMARD (%)": "{:+.1f}", "fit time (s)": "{:,.1f}"},
    na_rep="—",
))
print("tuning time per model (s): " + ", ".join(f"{key} {seconds:,.1f}" for key, seconds in RUNTIME["tuning (s)"].items()))

### 7.3 Scoreboard B: extremes

**Tail bins:** the edges are the `{1, 25, 75, 99} %` quantiles of the **actual** residual load over the
test window, computed once and shared by every row. MAE and bias are reported for the bottom bin
(`≤ P1`), the ordinary bin (`P25–P75`) and the top bin (`> P99`), once **binned by the actual** and
once **binned by the row's own forecast**. Regression to the mean makes a forecast look biased
towards the middle when binned by the actual, and towards the tails when binned by the forecast, so
only a tail difference that shows in **both** views counts.

**Day maximum and minimum:** the value error (`forecast − actual`) of each day's highest and lowest
hour, taken over the common hours only. A day counts if its common hours cover at least
`DAY_COMPLETENESS` of its expected hours.

In [ ]:
TAIL_LEVELS = [0.01, 0.25, 0.75, 0.99]
EDGES = actual[test_hours].quantile(TAIL_LEVELS)
P1, P25, P75, P99 = EDGES.to_numpy()
BINS = ["bottom", "ordinary", "top"]


def tail_bin(values):
    """bottom (<= P1), ordinary (P25..P75), top (> P99); anything else is outside the three bins."""
    return pd.Series(
        np.select([values <= P1, (values >= P25) & (values <= P75), values > P99], BINS, default="other"),
        index=values.index,
    )


print("Tail-bin edges: quantiles of the actual residual load over the test window (MWh)")
print("  " + ", ".join(f"P{level * 100:g} = {edge:,.0f}" for level, edge in EDGES.items()))

EXTREMES_RECORDS, tails = [], {"actual": {}, "forecast": {}}
for row, e in ERRORS.items():
    for view, values in [("actual", actual[COMMON]), ("forecast", ROWS[row][COMMON])]:
        bins = tail_bin(values)
        stats = {}
        for name in BINS:
            in_bin = e[bins == name]
            stats[(name, "MAE")], stats[(name, "bias")], stats[(name, "hours")] = in_bin.abs().mean(), in_bin.mean(), len(in_bin)
            EXTREMES_RECORDS.append((*row, "extremes", f"MAE_{name}_by_{view}", stats[(name, "MAE")], len(in_bin)))
            EXTREMES_RECORDS.append((*row, "extremes", f"bias_{name}_by_{view}", stats[(name, "bias")], len(in_bin)))
        tails[view][row] = stats

tail_format = {column: ("{:,.0f}" if column[1] == "hours" else "{:+,.0f}" if column[1] == "bias" else "{:,.0f}")
               for column in [(b, m) for b in BINS for m in ["MAE", "bias", "hours"]]}
for view in ["actual", "forecast"]:
    table = pd.DataFrame(tails[view]).T.rename_axis(["model", "split_method"])
    table.columns = pd.MultiIndex.from_tuples(table.columns)
    print(f"\nScoreboard B: error in the tails, binned by the {view} (MWh)")
    display(table.loc[SCOREBOARD_A.index].style.format(tail_format))

# Day maximum and minimum over the common hours, for days with enough of them.
common_day = COMMON.normalize()
common_per_day = pd.Series(1, index=COMMON).groupby(common_day).sum()
expected_per_day = SETTING["target_hours"].map(len).reindex(common_per_day.index)
DAYS_SCORED = common_per_day.index[(common_per_day >= DAY_COMPLETENESS * expected_per_day).to_numpy()]
scored = common_day.isin(DAYS_SCORED)

day_extremes = {}
for row in ERRORS:
    forecast, observed = ROWS[row][COMMON][scored], actual[COMMON][scored]
    by_day = common_day[scored]
    entry = {}
    for extreme, agg in [("day max", "max"), ("day min", "min")]:
        value_error = forecast.groupby(by_day).agg(agg) - observed.groupby(by_day).agg(agg)
        entry[(extreme, "MAE")], entry[(extreme, "bias")] = value_error.abs().mean(), value_error.mean()
        metric = extreme.replace(" ", "_")
        EXTREMES_RECORDS.append((*row, "extremes", f"MAE_{metric}", entry[(extreme, "MAE")], len(value_error)))
        EXTREMES_RECORDS.append((*row, "extremes", f"bias_{metric}", entry[(extreme, "bias")], len(value_error)))
    entry[("", "day_count")] = len(DAYS_SCORED)
    day_extremes[row] = entry
table = pd.DataFrame(day_extremes).T.rename_axis(["model", "split_method"])
table.columns = pd.MultiIndex.from_tuples(table.columns)
print(f"\nScoreboard B: value error of the day maximum and minimum (MWh), {len(DAYS_SCORED)} days")
display(table.loc[SCOREBOARD_A.index].style.format(
    {("day max", "MAE"): "{:,.0f}", ("day max", "bias"): "{:+,.0f}", ("day min", "MAE"): "{:,.0f}",
     ("day min", "bias"): "{:+,.0f}", ("", "day_count"): "{:,.0f}"}
))

# Tree extrapolation: a direct booster cannot forecast below the lowest target it was trained on.
static_training = hours_of(training_days(RUNS["test_static"]["fit_days"][0]))
lowest_test, lowest_train = actual[test_hours].idxmin(), actual[static_training].idxmin()
print(f"\nlowest actual residual load, test window         : {actual[lowest_test]:,.0f} MWh ({lowest_test:%Y-%m-%d %H:%M})")
print(f"lowest actual residual load, static fit's training : {actual[lowest_train]:,.0f} MWh ({lowest_train:%Y-%m-%d %H:%M}; "
      f"{static_training[0]:%Y-%m-%d} .. {static_training[-1]:%Y-%m-%d})")

### 7.4 Scoreboard C: intervals

Coverage is the share of common hours whose actual lies inside the row's band, against the nominal
level. SMARD has no band and therefore no row.

In [ ]:
INTERVAL_RECORDS, intervals = [], {}
for row, frame in FORECASTS.items():
    if row not in ROWS:
        continue
    band, observed = frame.loc[COMMON], actual[COMMON]
    inside = (band["lower"] <= observed) & (observed <= band["upper"])
    coverage, width = 100 * inside.mean(), (band["upper"] - band["lower"]).mean()
    intervals[row] = {"coverage (%)": coverage, "nominal (%)": 100 * INTERVAL["level"],
                      "coverage − nominal (pp)": coverage - 100 * INTERVAL["level"], "mean width (MWh)": width,
                      "hour_count": len(COMMON)}
    INTERVAL_RECORDS.append((*row, "intervals", "coverage_pct", coverage, len(COMMON)))
    INTERVAL_RECORDS.append((*row, "intervals", "mean_width", width, len(COMMON)))

SCOREBOARD_C = pd.DataFrame(intervals).T.rename_axis(["model", "split_method"])
print(f"Scoreboard C: {INTERVAL['level']:.0%} prediction intervals on {len(COMMON):,} common test hours")
display(SCOREBOARD_C.loc[[row for row in SCOREBOARD_A.index if row in SCOREBOARD_C.index]].style.format(
    {"coverage (%)": "{:.1f}", "nominal (%)": "{:.0f}", "coverage − nominal (pp)": "{:+.1f}",
     "mean width (MWh)": "{:,.0f}", "hour_count": "{:,.0f}"}
))

### 7.5 Static vs. rolling

The value of refitting: each model's MAE difference `rolling − static` on the common hours. A negative
difference means refitting helped. Both split methods share their first fit, so the first
`refit_every` of the test year cannot differ between them (§3.2).

In [ ]:
refit = {}
for key in MODELS:
    if (key, "static") in ERRORS and (key, "rolling") in ERRORS:
        static_mae, rolling_mae = ERRORS[(key, "static")].abs().mean(), ERRORS[(key, "rolling")].abs().mean()
        refit[key] = {"MAE static": static_mae, "MAE rolling": rolling_mae, "rolling − static": rolling_mae - static_mae,
                      "rolling − static (%)": 100 * (rolling_mae / static_mae - 1), "hour_count": len(COMMON)}
print("MAE difference rolling − static per model (MWh)")
display(pd.DataFrame(refit).T.style.format(
    {"MAE static": "{:,.0f}", "MAE rolling": "{:,.0f}", "rolling − static": "{:+,.0f}",
     "rolling − static (%)": "{:+.1f}", "hour_count": "{:,.0f}"}
) if refit else "both split methods are needed for this table")

### 7.6 Forecast plot with the band

The model in `PLOT_MODEL` under `PLOT_SPLIT_METHOD`. With `PLOT_MODEL = None` it is the registry model
with the lowest test MAE under that split method; `PLOT_SPLIT_METHOD` falls back to `static` when
rolling is switched off. Two panels: the local Monday–Sunday week of the test window that contains the
**highest** actual residual-load hour, and the one that contains the **lowest**, clipped to the test
window. Change `PLOT_MODEL` in §1.3 and re-run to see another model.

In [ ]:
registry_rows = [row for row in ERRORS if row[0] in MODELS]
plot_split = PLOT_SPLIT_METHOD if any(split == PLOT_SPLIT_METHOD for _, split in registry_rows) else "static"
candidates = {row: ERRORS[row].abs().mean() for row in registry_rows if row[1] == plot_split}
plot_key = PLOT_MODEL if PLOT_MODEL is not None else min(candidates, key=candidates.get)[0]
plot_row = (plot_key, plot_split)
if plot_row not in ROWS:
    raise ValueError(f"{plot_row} has no test-year forecast to plot; pick another PLOT_MODEL or PLOT_SPLIT_METHOD")

band, label, color = FORECASTS[plot_row], MODELS[plot_key]["label"], MODELS[plot_key]["color"]
test_actual = actual[test_hours]
peaks = {"highest": test_actual.idxmax(), "lowest": test_actual.idxmin()}


def week_of(stamp):
    """The test hours of the local Monday–Sunday week containing `stamp`, clipped to the test window."""
    monday = stamp.normalize() - pd.Timedelta(days=stamp.dayofweek)
    return test_hours[(test_hours >= monday) & (test_hours < monday + pd.Timedelta(days=7))]


print(f"plotted: {plot_key} / {plot_split}"
      + ("" if PLOT_MODEL else f"  (PLOT_MODEL = None -> lowest test MAE among registry rows under {plot_split})"))
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
for ax, (which, peak) in zip(axes, peaks.items()):
    hours = week_of(peak)
    print(f"  {which} actual hour {peak:%Y-%m-%d %H:%M} ({test_actual[peak]:,.0f} MWh) -> week {hours[0]:%Y-%m-%d} .. {hours[-1]:%Y-%m-%d}")
    ax.fill_between(hours, band.loc[hours, "lower"], band.loc[hours, "upper"], color=color, alpha=0.18, linewidth=0,
                    label=f"{label}: {INTERVAL['level']:.0%} band")
    ax.plot(hours, actual[hours], color=FIXED["actual"]["color"], linewidth=2, label=FIXED["actual"]["label"])
    ax.plot(hours, ROWS[SMARD_ROW][hours], color=FIXED["smard"]["color"], linewidth=1.6, linestyle="--", label=FIXED["smard"]["label"])
    ax.plot(hours, band.loc[hours, "forecast"], color=color, linewidth=1.8, label=f"{label} ({plot_split})")
    ax.axhline(0, color="0.75", linewidth=0.8, zorder=0)
    style_timeseries(ax, f"Week with the {which} actual residual load ({test_actual[peak]:,.0f} MWh, {peak:%a %d %b %Y %H:%M})", "MWh")
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_minor_locator(mdates.HourLocator(byhour=[6, 12, 18]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False)
fig.suptitle(f"{label} ({plot_split} split) vs. SMARD's day-ahead forecast, test year", fontsize=16)
fig.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

### 7.7 Self-check

In [ ]:
SCOREBOARD = pd.DataFrame(
    ACCURACY_RECORDS + EXTREMES_RECORDS + INTERVAL_RECORDS,
    columns=["model", "split_method", "table", "metric", "value", "count"],
)

assert COMMON.isin(test_hours).all() and len(COMMON) > 0
assert all(forecast[COMMON].notna().all() for forecast in ROWS.values()), "every row forecasts every common hour"
assert not SCOREBOARD.duplicated(["model", "split_method", "table", "metric"]).any()
smard_mae = SCOREBOARD.query("model == 'smard' and table == 'accuracy' and metric == 'MAE'")["value"].iloc[0]
assert abs(smard_mae - smard_errors.loc[COMMON, "err_residual_load"].abs().mean()) < 1e-9, "SMARD re-scored from its file"
assert not SCOREBOARD.query("model == 'smard' and table == 'intervals'").size, "SMARD has no band"
assert set(SCOREBOARD["model"]) <= set(MODELS) | {"smard", "seasonal_naive"}
assert set(SCOREBOARD["split_method"]) <= {"static", "rolling", "none"}
assert list(time_series.columns) == SERIES + DERIVED, "a section 7 cell persisted a column onto time_series"

print("section 7 self-check passed")
print(f"  {SCOREBOARD[['model', 'split_method']].drop_duplicates().shape[0]} rows scored on {len(COMMON):,} common hours; "
      f"{len(SCOREBOARD)} scoreboard values")

---

## 8 Export

Two files in `data/models/`, written **only when `EXPORT_ENABLED` is on** (default off, while the team
experiments). Both frames are always built in memory, so the scoreboards and the closing self-check
work with the toggle off.

| File | Grain | Columns |
|---|---|---|
| `model_forecast_errors_hourly.csv` | test hour × model × split method, for the registry rows and seasonal naive | `timestamp`, `model`, `split_method`, `residual_load`, `forecast`, `lower`, `upper`, `err_residual_load` |
| `model_scoreboard.csv` | long: model × split method × table × metric, incl. SMARD and seasonal naive | `model`, `split_method`, `table` (`accuracy` / `extremes` / `intervals`), `metric`, `value`, `count` (hours or days) |

- **SMARD is not in the hourly file:** its hourly values already live in
  `data/metrics/smard_forecast_errors_hourly.csv`. Timestamps are plain local time, like that file,
  so the two join on `timestamp`.
- Missing values stay empty. `err_residual_load = forecast − actual`, as everywhere in the project.
- Plain CSV (`sep=","`, `decimal="."`, UTF-8), like the other derived exports: a bare `pd.read_csv`
  reads them.
- `data/models/` is kept in git by an empty `.gitkeep`, and its CSVs are ignored by the
  `data/models/*.csv` rule. The notebook does **not** create the folder.

In [ ]:
EXPORT_DIR = DATA_DIR / "models"
EXPORT_PATHS = {
    "hourly": EXPORT_DIR / "model_forecast_errors_hourly.csv",
    "scoreboard": EXPORT_DIR / "model_scoreboard.csv",
}
HOURLY_COLUMNS = ["timestamp", "model", "split_method", "residual_load", "forecast", "lower", "upper", "err_residual_load"]
SCOREBOARD_COLUMNS = ["model", "split_method", "table", "metric", "value", "count"]

EXPORT_HOURLY = pd.concat(
    [
        pd.DataFrame({
            "timestamp": frame.index,
            "model": model,
            "split_method": split,
            "residual_load": actual[frame.index].to_numpy(),
            "forecast": frame["forecast"].to_numpy(),
            "lower": frame["lower"].to_numpy(),
            "upper": frame["upper"].to_numpy(),
            "err_residual_load": (frame["forecast"] - actual[frame.index]).to_numpy(),
        })
        for (model, split), frame in FORECASTS.items()
    ],
    ignore_index=True,
)[HOURLY_COLUMNS]
EXPORT_SCOREBOARD = SCOREBOARD[SCOREBOARD_COLUMNS]

print(f"model_forecast_errors_hourly (in memory): {len(EXPORT_HOURLY):,} rows = {len(test_hours):,} test hours x "
      f"{len(FORECASTS)} rows ({', '.join(f'{m} / {s}' for m, s in FORECASTS)})")
display(EXPORT_HOURLY.head(3))
print(f"model_scoreboard (in memory): {len(EXPORT_SCOREBOARD):,} values for "
      f"{EXPORT_SCOREBOARD[['model', 'split_method']].drop_duplicates().shape[0]} rows")
display(EXPORT_SCOREBOARD.head(3))

In [ ]:
if EXPORT_ENABLED:
    if not EXPORT_DIR.is_dir():
        raise FileNotFoundError(
            f"{EXPORT_DIR} does not exist. It is kept in git by an empty .gitkeep; restore it with "
            "`git checkout -- data/models/.gitkeep`. This notebook does not create it."
        )
    EXPORT_HOURLY.to_csv(EXPORT_PATHS["hourly"], index=False, encoding="utf-8", date_format="%Y-%m-%d %H:%M:%S")
    EXPORT_SCOREBOARD.to_csv(EXPORT_PATHS["scoreboard"], index=False, encoding="utf-8")
    print("export written:")
else:
    print("export skipped (EXPORT_ENABLED = False). With the toggle on, this cell would write:")
for name, path in EXPORT_PATHS.items():
    frame = EXPORT_HOURLY if name == "hourly" else EXPORT_SCOREBOARD
    print(f"  {path}  ({len(frame):,} rows)")